## 1. Notebook Scope (Human Evaluation)

This notebook is the human-evaluation-oriented analysis flow for stepwise reasoning (templates 41 and 42).
It builds auto matching artifacts, keeps reviewer override columns, and recalculates final metrics from adjudicated labels.


In [255]:
import transformers
import sentence_transformers
from sentence_transformers import SentenceTransformer

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("Sentence-BERT import OK")

transformers: 4.52.4
sentence-transformers: 5.1.2
Sentence-BERT import OK


In [256]:
!pip install pandas requests tqdm openpyxl

"pip" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


# 2. Model Run Setup (Claude Example)

### Quick Routing Check by Template

In [257]:
import pandas as pd
import re

# INPUT = "output_with_answers_recovered.csv"
# INPUT = "1_2_with_answers_claude-opus-4.6.csv"
INPUT = "3_2_to_3_5_clean_claude-opus-4.6_V2.csv"

COL_TEMPLATE = "template_id"
COL_GT = "ground_truth_answer"
COL_PRED = "model_answer"

from pathlib import Path
if not Path(INPUT).exists():
    raise FileNotFoundError(f"Claude Opus input CSV not found: {INPUT}")
df = pd.read_csv(INPUT)


### Imports and Configuration

In [258]:
import numpy as np
import pandas as pd
import re
from pathlib import Path
from typing import Any

# Claude section config (stepwise reasoning evaluation)
MODEL_NAME = "Claude Opus 4.6"
COL_MODEL_NAME = "model_name"
COL_TEMPLATE = "template_id"
COL_SCENE_ID = "scene_id"
COL_PARENT_RULE_ID = "ParentRuleID"
COL_FIGURE_ID = "rule_figure_id"
COL_GT = "ground_truth_answer"
COL_PRED = "model_answer"

STEPWISE_TEMPLATE_IDS = {41, 42}
RULE_CLASSIFICATION_LAYER = "stepwise_reasoning"
RULE_UNDERSTANDING_LAYER = "n/a"

GT_REASONING_LOOKUP_CSV = r"C:\Users\ritaMZ\WebstormProjects\thesis\backend\03_04_reasonLLMjudgeCalibr.csv"
GT_LOOKUP_AUDIT_CSV = "3_2_to_3_5_gt_lookup_audit.csv"

# Matching and triage thresholds.
RULE_WEAK_MATCH_THRESHOLD = 0.58
RULE_MATCH_THRESHOLD = 0.71
RULE_STRONG_MATCH_THRESHOLD = 0.82
MATCH_THRESHOLD = RULE_MATCH_THRESHOLD
STRONG_MATCH_THRESHOLD = RULE_STRONG_MATCH_THRESHOLD
RULE_AUTO_ACCEPT_THRESHOLD = RULE_STRONG_MATCH_THRESHOLD
RULE_AUTO_REJECT_THRESHOLD = RULE_WEAK_MATCH_THRESHOLD
ORDER_ANCHOR_POLICY = "min_pred_position"

# Review/export files for 3_2_to_3_5 stepwise evaluation.
RULE_CLASSIFICATION_CANDIDATE_CSV = "3_2_to_3_5_step_candidate_matches.csv"
RULE_ALL_PAIRWISE_CSV = "3_2_to_3_5_all_pairwise_similarity.csv"
RULE_GT_COVERAGE_CSV = "3_2_to_3_5_gt_coverage_summary.csv"
RULE_SAMPLE_LEVEL_METRICS_CSV = "3_2_to_3_5_sample_level_metrics.csv"


In [259]:
import math
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

RULE_UNDERSTANDING_TITLE = "Stepwise Reasoning Evaluation (3_2 to 3_5)"
RULE_UNDERSTANDING_SUMMARY_CSV = "3_2_to_3_5_stepwise_summary_table.csv"
RULE_UNDERSTANDING_SUMMARY_XLSX = "3_2_to_3_5_stepwise_summary_table.xlsx"

RULE_MODEL_INPUTS = {
    "Claude Opus 4.6": "3_2_to_3_5_clean_claude-opus-4.6_V2.csv",
    "Gemini 3 Pro Preview": "3_2_to_3_5_clean_gemini_3_pro_prev_V2.csv",
    "GPT-5.2": "3_2_to_3_5_clean_gpt-5_2_V2.csv",
}
RULE_MODEL_COLUMNS = ["Claude Opus 4.6", "Gemini 3 Pro Preview", "GPT-5.2"]

# Input schema in this notebook is standardized to:
# - ground_truth_answer
# - model_answer


### Prepare Base DataFrame and Normalize Fields

In [260]:
def _ensure_rule_columns(df_in: pd.DataFrame) -> pd.DataFrame:
    df_local = df_in.copy()

    required = [COL_PRED]
    missing = [c for c in required if c not in df_local.columns]
    if missing:
        raise KeyError(f"Missing required columns for stepwise evaluation: {missing}")

    if COL_GT not in df_local.columns:
        df_local[COL_GT] = ""
    if COL_MODEL_NAME not in df_local.columns:
        df_local[COL_MODEL_NAME] = MODEL_NAME

    df_local[COL_MODEL_NAME] = df_local[COL_MODEL_NAME].fillna(MODEL_NAME).astype(str).str.strip()
    df_local[COL_GT] = df_local[COL_GT].fillna("").astype(str)
    df_local[COL_PRED] = df_local[COL_PRED].fillna("").astype(str)

    if "generated_question_id" not in df_local.columns:
        df_local["generated_question_id"] = pd.NA
    if "row_index" not in df_local.columns:
        df_local["row_index"] = df_local.index
    if "question_text" not in df_local.columns and "question" in df_local.columns:
        df_local["question_text"] = df_local["question"].fillna("").astype(str)
    elif "question_text" not in df_local.columns:
        df_local["question_text"] = ""

    if COL_TEMPLATE in df_local.columns:
        df_local["_template_id_norm"] = pd.to_numeric(df_local[COL_TEMPLATE], errors="coerce")
    else:
        df_local["_template_id_norm"] = np.nan

    return df_local


# Raw pre-check routing. Final stepwise routing is rebuilt later from _prepare_rule_eval_df(INPUT),
# which attaches canonical GT reasoning and GT match statuses.
df_eval_raw = _ensure_rule_columns(df)
rule_classification_mask_raw = df_eval_raw["_template_id_norm"].isin(list(STEPWISE_TEMPLATE_IDS))

print("Initial input routing (raw, before GT lookup):")
print(f"- Stepwise template rows {sorted(STEPWISE_TEMPLATE_IDS)}: {int(rule_classification_mask_raw.sum())}")
print(f"- Other rows: {int(len(df_eval_raw) - rule_classification_mask_raw.sum())}")


Initial input routing (raw, before GT lookup):
- Stepwise template rows [41, 42]: 140
- Other rows: 0


### Parsing Helpers

In [261]:
import numpy as np
import re


_STEP_HEADER_RE = re.compile(
    r"(?im)^\s*step\s+(?P<num>\d+)\s*(?:[\.\:\-\)\]]\s*)?(?P<title>[^\n]*)"
)

_FINAL_LABEL_MATCH_THRESHOLD = 0.62

_FINAL_STEP_REFERENCE_LABELS = [
    "final step",
    "final evaluation",
    "final assessment",
    "final conclusion",
    "conclusion",
    "overall conclusion",
    "final compliance verdict",
    "compliance verdict",
]


def _norm_text(v: Any) -> str:
    s = "" if pd.isna(v) else str(v)
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


def _normalize_reasoning_text(raw_text: Any) -> str:
    s = "" if pd.isna(raw_text) else str(raw_text)
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+\n", "\n", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def _clean_step_title(title: str) -> str:
    t = "" if title is None else str(title)
    t = re.sub(r"\s+", " ", t).strip()
    t = t.strip("-:;.")
    return t


def _fallback_split_numbered_items(raw_text: Any) -> list[dict[str, Any]]:
    """Fallback parser for non-Step formats (e.g., `1. ...` lists)."""
    s = _normalize_reasoning_text(raw_text)
    if not s:
        return []

    numbered = list(re.finditer(r"(?m)(^|\n)\s*(\d+)[\.)]\s*", s))
    if not numbered:
        lines = [ln.strip() for ln in s.split("\n") if ln.strip()]
        parts = []
        for ln in lines:
            ln2 = re.sub(r"^\s*(?:\d+[\.)]|[-*?])\s*", "", ln).strip(" \t-:;")
            if ln2:
                parts.append(ln2)
        if len(parts) == 1 and ";" in parts[0]:
            semi_parts = [p.strip() for p in parts[0].split(";") if p.strip()]
            if len(semi_parts) > 1:
                parts = semi_parts
        out = []
        for pos, part in enumerate(parts, start=1):
            out.append(
                {
                    "step_id": f"S_{pos}",
                    "step_number": pos,
                    "step_position": pos,
                    "step_label": "",
                    "step_text": part,
                    "full_step_text": part,
                }
            )
        return out

    out = []
    for pos, m in enumerate(numbered, start=1):
        start = m.end()
        end = numbered[pos].start() if pos < len(numbered) else len(s)
        body = s[start:end].strip(" \t\n-:;")
        step_no = int(m.group(2))
        full = body if body else f"{step_no}."
        out.append(
            {
                "step_id": f"S_{step_no}",
                "step_number": step_no,
                "step_position": pos,
                "step_label": "",
                "step_text": body,
                "full_step_text": full,
            }
        )
    return out


def _parse_reasoning_steps(raw_text: Any) -> list[dict[str, Any]]:
    """Parse full reasoning step blocks by valid `Step N...` headers.

    One parsed item = one full step block from a valid header to the next valid header (or end of text).
    """
    text = _normalize_reasoning_text(raw_text)
    if not text:
        return []

    headers = list(_STEP_HEADER_RE.finditer(text))
    if not headers:
        return _fallback_split_numbered_items(text)

    parsed = []
    for pos, h in enumerate(headers, start=1):
        block_start = h.start()
        block_end = headers[pos].start() if pos < len(headers) else len(text)
        block = text[block_start:block_end].strip()

        header_line_end = text.find("\n", h.start())
        if header_line_end == -1 or header_line_end > block_end:
            header_line_end = block_end

        raw_title = h.group("title") if h.group("title") is not None else ""
        step_label = _clean_step_title(raw_title)

        body = text[header_line_end:block_end].strip() if header_line_end < block_end else ""
        step_number = int(h.group("num"))

        parsed.append(
            {
                "step_id": f"S_{step_number}",
                "step_number": step_number,
                "step_position": pos,
                "step_label": step_label,
                "step_text": body,
                "full_step_text": block,
            }
        )

    return parsed


def _step_text_for_matching(step: dict[str, Any]) -> str:
    """Build stable semantic text for matching while preserving full-step boundaries."""
    label = "" if pd.isna(step.get("step_label", "")) else str(step.get("step_label", "")).strip()
    body = "" if pd.isna(step.get("step_text", "")) else str(step.get("step_text", "")).strip()
    full = "" if pd.isna(step.get("full_step_text", "")) else str(step.get("full_step_text", "")).strip()

    parts = [p for p in [label, body] if p]
    if parts:
        return " ".join(parts).strip()
    return full


def _split_steps(raw_text: Any) -> list[str]:
    """Return one semantic string per full parsed step block."""
    steps = _parse_reasoning_steps(raw_text)
    if not steps:
        return []

    out = []
    seen = set()
    for st in steps:
        text = _step_text_for_matching(st)
        key = _norm_text(text)
        if key and key not in seen:
            out.append(text)
            seen.add(key)
    return out


def _split_atomic_rules(raw_text: Any) -> list[str]:
    """Deprecated alias kept for backward compatibility."""
    return _split_steps(raw_text)


def _step_label_for_final_detection(step: dict[str, Any]) -> str:
    label = "" if pd.isna(step.get("step_label", "")) else str(step.get("step_label", "")).strip()
    if label:
        return label
    # Fallback to first sentence/line from step text when explicit label is missing.
    txt = "" if pd.isna(step.get("step_text", "")) else str(step.get("step_text", "")).strip()
    if not txt:
        txt = "" if pd.isna(step.get("full_step_text", "")) else str(step.get("full_step_text", "")).strip()
    first_line = txt.split("\n", 1)[0].strip() if txt else ""
    return first_line[:220]


def _contains_final_step_keyword(label: str) -> bool:
    t = _norm_text(label)
    if not t:
        return False
    for ref in _FINAL_STEP_REFERENCE_LABELS:
        if ref in t:
            return True
    return False


def _final_step_label_similarity(label: str) -> float:
    """Sentence-BERT similarity of step label vs canonical final-step label phrases."""
    lbl = _norm_text(label)
    if not lbl:
        return np.nan
    if _contains_final_step_keyword(lbl):
        return 1.0

    scorer = globals().get("_safe_sbert_pairwise_scores")
    if not callable(scorer):
        return np.nan
    try:
        scores = scorer([lbl], _FINAL_STEP_REFERENCE_LABELS)
        if scores.size == 0:
            return np.nan
        return float(np.nanmax(scores[0]))
    except Exception:
        return np.nan

def _final_label_pair_similarity(gt_label: str, pred_label: str) -> float:
    """Semantic similarity between GT and predicted final-step labels."""
    g = _norm_text(gt_label)
    p = _norm_text(pred_label)
    if not g or not p:
        return np.nan
    if g == p:
        return 1.0
    if g in p or p in g:
        return 0.98

    key_terms = {"final", "conclusion", "assessment", "evaluation", "verdict", "check"}
    g_terms = set(g.split())
    p_terms = set(p.split())
    if (g_terms & p_terms & key_terms):
        lexical_score = 0.85
    else:
        lexical_score = np.nan

    scorer = globals().get("_safe_sbert_pairwise_scores")
    sbert_score = np.nan
    if callable(scorer):
        try:
            m = scorer([g], [p])
            if m.size > 0:
                sbert_score = float(m[0, 0])
        except Exception:
            sbert_score = np.nan

    if pd.isna(lexical_score) and pd.isna(sbert_score):
        return np.nan
    if pd.isna(lexical_score):
        return sbert_score
    if pd.isna(sbert_score):
        return lexical_score
    return float(max(lexical_score, sbert_score))


def _match_final_step_by_label(
    gt_step: dict[str, Any],
    pred_steps: list[dict[str, Any]],
    pred_final_indices: set[int],
    similarity_threshold: float = _FINAL_LABEL_MATCH_THRESHOLD,
) -> tuple[list[int], float]:
    """Fallback label-level matching for final step only."""
    if not pred_steps or not pred_final_indices:
        return [], np.nan

    gt_label = _step_label_for_final_detection(gt_step)
    if not gt_label:
        return [], np.nan

    candidates = []
    for idx in sorted(pred_final_indices):
        if idx < 0 or idx >= len(pred_steps):
            continue
        pred_label = _step_label_for_final_detection(pred_steps[idx])
        sim = _final_label_pair_similarity(gt_label, pred_label)
        if not pd.isna(sim):
            candidates.append((idx, float(sim)))

    if not candidates:
        return [], np.nan

    best_idx, best_sim = max(candidates, key=lambda x: x[1])
    if float(best_sim) >= float(similarity_threshold):
        return [int(best_idx)], float(best_sim)
    return [], float(best_sim)


def _detect_final_step_indices(steps: list[dict[str, Any]], similarity_threshold: float = 0.62) -> set[int]:
    """Find final-step indices with explicit guardrails.

    Rule: choose a detected final step by label match/similarity; otherwise fallback to the last step.
    """
    if not steps:
        return set()

    candidates = []
    for i, st in enumerate(steps):
        label = _step_label_for_final_detection(st)
        has_kw = _contains_final_step_keyword(label)
        sim = _final_step_label_similarity(label)
        if has_kw or (not pd.isna(sim) and float(sim) >= float(similarity_threshold)):
            candidates.append((i, float(sim) if not pd.isna(sim) else -1.0, has_kw))

    if candidates:
        # Prefer the latest detected final-like step to align with "final step is usually last" policy.
        best_idx = max(candidates, key=lambda x: x[0])[0]
        return {int(best_idx)}

    return {int(len(steps) - 1)}


def _apply_final_step_incompatibility_guard(
    col_scores: np.ndarray,
    gt_idx: int,
    gt_final_indices: set[int],
    pred_final_indices: set[int],
) -> np.ndarray:
    """Prevent accidental cross-matching between final verdict step and non-final steps."""
    if col_scores.size == 0:
        return col_scores

    guarded = col_scores.astype(float).copy()
    n_pred = len(guarded)

    if gt_idx in gt_final_indices:
        # Final GT step can only match final predicted step(s).
        invalid = [p for p in range(n_pred) if p not in pred_final_indices]
    else:
        # Non-final GT steps cannot consume final predicted step(s).
        invalid = [p for p in pred_final_indices if 0 <= p < n_pred]

    for p in invalid:
        guarded[p] = np.nan
    return guarded


def _safe_div(num: float, den: float) -> float:
    return float(num / den) if den else np.nan


def _serialize_ids(ids: list[int], prefix: str) -> str:
    if not ids:
        return ""
    return ",".join(f"{prefix}_{i}" for i in ids)


def _serialize_positions(ids: list[int]) -> str:
    if not ids:
        return ""
    return ",".join(str(i + 1) for i in ids)


def _serialize_texts(items: list[str]) -> str:
    if not items:
        return ""
    return " | ".join(items)


def _triage_zone(score: float, accept_threshold: float, reject_threshold: float) -> str:
    if pd.isna(score):
        return "auto_reject"
    if score >= accept_threshold:
        return "auto_accept"
    if score <= reject_threshold:
        return "auto_reject"
    return "uncertain"


def _needs_human_review(zone: str) -> int:
    return int(zone == "uncertain")


def _coverage_from_scores(
    col_scores: np.ndarray,
    match_threshold: float,
    strong_threshold: float,
) -> tuple[list[int], str, str]:
    if col_scores.size == 0:
        return [], "missing", ""

    matched_idx = [i for i, s in enumerate(col_scores) if (not pd.isna(s)) and float(s) >= float(match_threshold)]
    m = len(matched_idx)
    if m == 0:
        return matched_idx, "missing", ""
    if m == 1:
        q = "strong" if float(col_scores[matched_idx[0]]) >= float(strong_threshold) else "weak"
        return matched_idx, "exact", q

    all_strong = all(float(col_scores[i]) >= float(strong_threshold) for i in matched_idx)
    return matched_idx, "split", ("strong" if all_strong else "weak")


def _is_blank_series(sr: pd.Series) -> pd.Series:
    return sr.isna() | (sr.astype(str).str.strip() == "")


# Parse serialized predicted IDs such as "P_0,P_2" into integer indices [0, 2].
def _parse_pred_ids(value: Any) -> list[int]:
    s = "" if pd.isna(value) else str(value)
    if not s.strip():
        return []

    out = []
    for token in s.split(","):
        t = token.strip().upper()
        if t.startswith("P_"):
            t = t[2:]
        elif t.startswith("P"):
            t = t[1:]
        if t.isdigit():
            out.append(int(t))
    return out


#### Parser Sanity Checks (Step Extraction)

Validates that GT and prediction reasoning are split into comparable step units.
These checks reduce review noise caused by formatting/parsing drift.


In [262]:
PARSER_EXAMPLES = {
    "example_1": """Step 1. Task understanding
The rule requires a grab bar near the toilet.
Step 2. Scene inspection
The BIM scene shows no grab bar.
Step 3. Conclusion
Therefore, the design is non-compliant.""",
    "example_2": """Step 1 Explanation: Identify the relevant rule.
The answer must check accessibility compliance.
Step 2 Explanation: Compare the scene with the rule.
The sink height exceeds the allowed range.""",
    "example_3": """Step 1: Task understanding - determine what is being checked.
The model identifies the applicable requirement.
Step 2: Compare observation with rule.
The observed dimension is smaller than required.""",
}

preview_rows = []
for ex_name, ex_text in PARSER_EXAMPLES.items():
    parsed = _parse_reasoning_steps(ex_text)
    for st in parsed:
        preview_rows.append(
            {
                "example": ex_name,
                "step_id": st.get("step_id"),
                "step_number": st.get("step_number"),
                "step_position": st.get("step_position"),
                "step_label": st.get("step_label"),
                "step_text": st.get("step_text"),
                "full_step_text": st.get("full_step_text"),
            }
        )

parser_preview_df = pd.DataFrame(preview_rows)
display(parser_preview_df)

# Minimal acceptance checks.
assert len(_parse_reasoning_steps(PARSER_EXAMPLES["example_1"])) == 3
assert len(_parse_reasoning_steps(PARSER_EXAMPLES["example_2"])) == 2
assert len(_parse_reasoning_steps(PARSER_EXAMPLES["example_3"])) == 2
assert not any(
    _norm_text(x).strip() == "step"
    for x in _split_steps(PARSER_EXAMPLES["example_3"])
)
print("Parser sanity checks passed.")


,example,step_id,step_number,step_position,step_label,step_text,full_step_text
0,example_1,S_1,1,1,Task understanding,The rule requires a grab bar near the toilet.,Step 1. Task understanding\nThe rule requires ...
1,example_1,S_2,2,2,Scene inspection,The BIM scene shows no grab bar.,Step 2. Scene inspection\nThe BIM scene shows ...
2,example_1,S_3,3,3,Conclusion,"Therefore, the design is non-compliant.","Step 3. Conclusion\nTherefore, the design is n..."
3,example_2,S_1,1,1,Explanation: Identify the relevant rule,The answer must check accessibility compliance.,Step 1 Explanation: Identify the relevant rule...
4,example_2,S_2,2,2,Explanation: Compare the scene with the rule,The sink height exceeds the allowed range.,Step 2 Explanation: Compare the scene with the...
5,example_3,S_1,1,1,Task understanding - determine what is being c...,The model identifies the applicable requirement.,Step 1: Task understanding - determine what is...
6,example_3,S_2,2,2,Compare observation with rule,The observed dimension is smaller than required.,Step 2: Compare observation with rule.\nThe ob...


Parser sanity checks passed.


#### Matching and Aggregation Helpers

Computes SBERT-based matching, final-step guardrails, and coverage labels used in review tables.
Coverage labels: `exact` / `split` / `missing`; quality labels: `strong` / `weak` / `wrong`.


In [263]:
def _greedy_one_to_one_assignment(scores: np.ndarray) -> dict[int, int]:
    """Greedy one-to-one assignment on similarity matrix (rows=pred, cols=gt)."""
    assignments: dict[int, int] = {}
    if scores.size == 0:
        return assignments

    n_pred, n_gt = scores.shape
    if n_pred == 0 or n_gt == 0:
        return assignments

    triples = []
    for pred_i in range(n_pred):
        for gt_i in range(n_gt):
            val = float(scores[pred_i, gt_i])
            if np.isnan(val):
                continue
            triples.append((val, pred_i, gt_i))
    triples.sort(key=lambda x: x[0], reverse=True)

    used_pred = set()
    used_gt = set()
    for _, pred_i, gt_i in triples:
        if pred_i in used_pred or gt_i in used_gt:
            continue
        assignments[pred_i] = gt_i
        used_pred.add(pred_i)
        used_gt.add(gt_i)
    return assignments


def _is_blank_value(v) -> bool:
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except Exception:
        pass
    return isinstance(v, str) and v.strip() == ""


def _sample_key(generated_question_id, row_index) -> str:
    gq = "" if _is_blank_value(generated_question_id) else str(generated_question_id)
    if _is_blank_value(row_index):
        ri = ""
    else:
        try:
            ri = str(int(float(row_index)))
        except Exception:
            ri = str(row_index)
    return f"{gq}||{ri}"


def _tokenize_text(s: str) -> list[str]:
    t = _norm_text(s)
    return t.split() if t else []



def _aggregate_rate(sample_df: pd.DataFrame, num_col: str, den_col: str) -> float:
    if sample_df.empty:
        return np.nan
    den = float(sample_df[den_col].sum())
    if den <= 0:
        return np.nan
    num = float(sample_df[num_col].sum())
    return float(num / den)


def _dataset_text_overall(n_questions: int) -> str:
    return f"N = {int(n_questions)} questions"


def _dataset_text_classification(n_questions: int, gt_steps: int | None) -> str:
    if gt_steps is None:
        return f"Q = {int(n_questions)}"
    return f"Q = {int(n_questions)}, GT steps = {int(gt_steps)}"


def _ngrams(tokens: list[str], n: int):
    if len(tokens) < n or n <= 0:
        return []
    return [tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)]


#### Metric Helpers (How Metrics Are Computed)

Sample-level metrics are computed from final GT-centric labels:
- `coverage_recall = gt_covered_count / gt_total_steps`
- `split_step_rate = gt_split_count / gt_total_steps`
- `missing_step_rate = gt_missing_count / gt_total_steps`
- `weak_match_rate = gt_weak_count / gt_total_steps`
- `extra_step_rate = pred_extra_count / pred_total_steps`

Text-similarity diagnostics:
- `overall Sentence-BERT similarity`: mean of `best_similarity` over GT rows
- `ROUGE-L (avg per step)`: mean ROUGE-L F1 over matched GT/pred step text pairs
- `order_preservation_score`: consistency of predicted step order against GT order.


In [264]:
def _get_sbert_model():
    model_sbert = globals().get("_SCENE_SBERT_MODEL")
    if model_sbert is None:
        from sentence_transformers import SentenceTransformer

        model_sbert = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
        globals()["_SCENE_SBERT_MODEL"] = model_sbert
    return model_sbert


def _sbert_pairwise_scores(pred_texts: list[str], gt_texts: list[str]) -> np.ndarray:
    """Pairwise SBERT cosine similarity: rows=predictions, cols=ground-truth."""
    if len(pred_texts) == 0 or len(gt_texts) == 0:
        return np.empty((len(pred_texts), len(gt_texts)), dtype=float)

    try:
        from sentence_transformers import util

        model_sbert = _get_sbert_model()

        emb_pred = model_sbert.encode(
            pred_texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        emb_gt = model_sbert.encode(
            gt_texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        scores = util.cos_sim(emb_pred, emb_gt).cpu().numpy()
        return scores.astype(float)
    except Exception as e:
        print(f"Sentence-BERT pairwise similarity failed: {e}")
        return np.empty((len(pred_texts), len(gt_texts)), dtype=float)

def _safe_sbert_pairwise_scores(pred_texts: list[str], gt_texts: list[str]) -> np.ndarray:
    if len(pred_texts) == 0 or len(gt_texts) == 0:
        return np.empty((len(pred_texts), len(gt_texts)), dtype=float)
    try:
        from sentence_transformers import SentenceTransformer, util

        model = globals().get("_RULE_SBERT_MODEL")
        if model is None:
            model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
            globals()["_RULE_SBERT_MODEL"] = model
        emb_pred = model.encode(
            pred_texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        emb_gt = model.encode(
            gt_texts,
            convert_to_tensor=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        return util.cos_sim(emb_pred, emb_gt).cpu().numpy().astype(float)
    except Exception as e:
        if "_RULE_SBERT_WARNED" not in globals():
            print(f"Sentence-BERT unavailable for rule-understanding metrics: {e}")
            globals()["_RULE_SBERT_WARNED"] = True
        return np.full((len(pred_texts), len(gt_texts)), np.nan, dtype=float)


def _mean_sbert_similarity_local(sub: pd.DataFrame) -> float:
    if sub.empty or "ground_truth_answer" not in sub.columns or "model_answer" not in sub.columns:
        return np.nan
    gt = sub["ground_truth_answer"].map(_norm_text)
    pred = sub["model_answer"].map(_norm_text)
    valid = (gt != "") & (pred != "")
    if valid.sum() == 0:
        return np.nan
    scores = _safe_sbert_pairwise_scores(pred[valid].tolist(), gt[valid].tolist())
    if scores.size == 0:
        return np.nan
    diag = np.diag(scores)
    if len(diag) == 0:
        return np.nan
    if np.isnan(diag).all():
        return np.nan
    return float(np.nanmean(diag))


def _lcs_len(a: list[str], b: list[str]) -> int:
    if not a or not b:
        return 0
    m = len(b)
    prev = [0] * (m + 1)
    for i in range(1, len(a) + 1):
        curr = [0] * (m + 1)
        ai = a[i - 1]
        for j in range(1, m + 1):
            if ai == b[j - 1]:
                curr[j] = prev[j - 1] + 1
            else:
                curr[j] = max(prev[j], curr[j - 1])
        prev = curr
    return prev[m]


def _rouge_l_f1(reference_text: str, predicted_text: str) -> float:
    ref = _tokenize_text(reference_text)
    pred = _tokenize_text(predicted_text)
    if not ref and not pred:
        return np.nan
    if not ref or not pred:
        return 0.0
    lcs = _lcs_len(ref, pred)
    if lcs == 0:
        return 0.0
    recall = lcs / len(ref)
    precision = lcs / len(pred)
    if recall + precision == 0:
        return 0.0
    return float(2 * recall * precision / (recall + precision))


def _bleu2(reference_text: str, predicted_text: str, eps: float = 1e-12) -> float:
    ref = _tokenize_text(reference_text)
    pred = _tokenize_text(predicted_text)
    if not ref and not pred:
        return np.nan
    if not pred:
        return 0.0
    if not ref:
        return 0.0

    precisions = []
    for n in [1, 2]:
        pred_counts = Counter(_ngrams(pred, n))
        ref_counts = Counter(_ngrams(ref, n))
        total = sum(pred_counts.values())
        if total == 0:
            precisions.append(eps)
            continue
        clipped = sum(min(cnt, ref_counts[ng]) for ng, cnt in pred_counts.items())
        p_n = clipped / total
        precisions.append(max(p_n, eps))

    c = len(pred)
    r = len(ref)
    bp = 1.0 if c > r else math.exp(1.0 - (r / max(c, 1)))
    score = bp * math.exp(sum(math.log(p) for p in precisions) / 2.0)
    return float(score)

#### Data Preparation and Processing Functions

Loads canonical GT lookup context and marks each row as auto-eligible or manual-review candidate.
This split is central for human evaluation because non-eligible rows are explicitly audited.


In [265]:
def _normalize_lookup_int(value: Any) -> float:
    if _is_blank_value(value):
        return np.nan
    try:
        return float(int(float(value)))
    except Exception:
        return np.nan


def _normalize_figure_lookup(value: Any) -> float:
    """Treat figure_id=0 and blank/None as the same missing value for GT retrieval matching."""
    x = _normalize_lookup_int(value)
    if pd.isna(x):
        return np.nan
    # In this dataset, 0 means no figure (same semantic as NULL/None).
    if float(x) == 0.0:
        return np.nan
    return x


def _extract_parent_rule_id(row: pd.Series) -> float:
    direct = row.get("ParentRuleID", np.nan)
    direct_num = _normalize_lookup_int(direct)
    if not pd.isna(direct_num):
        return direct_num

    alt = row.get("parent_rule_id", np.nan)
    alt_num = _normalize_lookup_int(alt)
    if not pd.isna(alt_num):
        return alt_num

    # Fallback: parse leading numeric parent rule ID from text labels such as "2. ...".
    txt = row.get("ClassificationParent", "")
    txt = "" if pd.isna(txt) else str(txt)
    m = re.search(r"(\d+)", txt)
    if m:
        return float(int(m.group(1)))
    return np.nan


def _load_gt_reasoning_lookup(lookup_csv: str) -> pd.DataFrame:
    p = Path(lookup_csv)
    if not p.exists():
        raise FileNotFoundError(f"GT reasoning lookup CSV not found: {lookup_csv}")

    gt_raw = pd.read_csv(p, sep=";", engine="python")
    required = ["ground_truth_answer", "scene_id", "parent_rule_id", "figure_id"]
    miss = [c for c in required if c not in gt_raw.columns]
    if miss:
        raise KeyError(f"Missing required GT lookup columns: {miss}")

    gt = gt_raw.copy()
    gt["scene_id_norm"] = gt["scene_id"].apply(_normalize_lookup_int)
    gt["parent_rule_id_norm"] = gt["parent_rule_id"].apply(_normalize_lookup_int)
    gt["figure_id_norm"] = gt["figure_id"].apply(_normalize_figure_lookup)
    gt["ground_truth_answer"] = gt["ground_truth_answer"].fillna("").astype(str)
    gt["_gt_text_norm"] = gt["ground_truth_answer"].map(_norm_text)
    return gt


def _attach_gt_reasoning(df_in: pd.DataFrame, gt_lookup_df: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    df["scene_id_norm"] = df.get("scene_id", np.nan).apply(_normalize_lookup_int)
    df["parent_rule_id_norm"] = df.apply(_extract_parent_rule_id, axis=1)
    df["figure_id_norm"] = df.get("rule_figure_id", np.nan).apply(_normalize_figure_lookup)

    def _key_part(v: Any):
        # Critical for matching: treat missing values as equal sentinels in lookup keys.
        return None if pd.isna(v) else float(v)

    def _build_key(scene_v: Any, parent_v: Any, figure_v: Any) -> tuple[Any, Any, Any]:
        return (_key_part(scene_v), _key_part(parent_v), _key_part(figure_v))

    gt = gt_lookup_df.copy()
    gt["_lookup_key"] = gt.apply(
        lambda r: _build_key(r.get("scene_id_norm"), r.get("parent_rule_id_norm"), r.get("figure_id_norm")),
        axis=1,
    )

    # Keep only non-blank GT text candidates for retrieval.
    gt_nonblank = gt[gt.get("_gt_text_norm", "") != ""].copy()

    gt_lookup_map: dict[tuple[Any, Any, Any], pd.DataFrame] = {}
    for key, sub in gt_nonblank.groupby("_lookup_key", dropna=False):
        gt_lookup_map[key] = sub.copy()

    statuses = []
    gt_texts = []
    gt_source_ids = []

    for _, row in df.iterrows():
        key = _build_key(row.get("scene_id_norm"), row.get("parent_rule_id_norm"), row.get("figure_id_norm"))
        sub = gt_lookup_map.get(key)
        if sub is None or sub.empty:
            statuses.append("no_match")
            gt_texts.append("")
            gt_source_ids.append(pd.NA)
            continue

        unique_texts = sub[["_gt_text_norm", "ground_truth_answer"]].drop_duplicates(subset=["_gt_text_norm"])
        if len(unique_texts) == 1:
            gt_text = str(unique_texts.iloc[0]["ground_truth_answer"])
            statuses.append("unique_match")
            gt_texts.append(gt_text)
            src_row = sub.iloc[0]
            gt_source_ids.append(src_row.get("reason_gt_id", pd.NA))
        else:
            statuses.append("multiple_matches")
            gt_texts.append("")
            gt_source_ids.append(pd.NA)

    df["gt_match_status"] = statuses
    df["gt_source_row_id"] = gt_source_ids
    df["gt_reasoning_text"] = gt_texts
    df[COL_GT] = df["gt_reasoning_text"].fillna("")
    df["gt_total_steps"] = df[COL_GT].apply(lambda x: len(_split_steps(x)))
    return df

def _prepare_rule_eval_df(input_path: str, model_name: str | None = None) -> tuple[pd.DataFrame, bool]:
    p = Path(input_path)
    if not p.exists():
        return pd.DataFrame(), True

    df_raw = pd.read_csv(p)
    if "model_answer" not in df_raw.columns:
        raise KeyError(f"Missing required column 'model_answer' in {input_path}")

    df = df_raw.copy()
    if "ground_truth_answer" not in df.columns:
        df["ground_truth_answer"] = ""
    if "model_name" not in df.columns:
        df["model_name"] = model_name or MODEL_NAME
    if "template_id" not in df.columns:
        df["template_id"] = np.nan

    df["ground_truth_answer"] = df["ground_truth_answer"].fillna("").astype(str)
    df["model_answer"] = df["model_answer"].fillna("").astype(str)
    df["model_name"] = df["model_name"].fillna(model_name or MODEL_NAME).astype(str).str.strip()
    df["_template_id_norm"] = pd.to_numeric(df["template_id"], errors="coerce")

    if "generated_question_id" not in df.columns:
        df["generated_question_id"] = pd.NA
    if "row_index" not in df.columns:
        df["row_index"] = df.index
    if "question_text" not in df.columns:
        if "question" in df.columns:
            df["question_text"] = df["question"].fillna("").astype(str)
        else:
            df["question_text"] = ""
    if "scene_id" not in df.columns:
        df["scene_id"] = np.nan
    if "ParentRuleID" not in df.columns and "parent_rule_id" not in df.columns:
        df["ParentRuleID"] = np.nan
    if "rule_figure_id" not in df.columns and "figure_id" not in df.columns:
        df["rule_figure_id"] = np.nan

    gt_lookup_df = _load_gt_reasoning_lookup(GT_REASONING_LOOKUP_CSV)
    df = _attach_gt_reasoning(df, gt_lookup_df)
    df["_is_stepwise_target"] = df["_template_id_norm"].isin(list(STEPWISE_TEMPLATE_IDS))
    df["_is_auto_eligible"] = df["_is_stepwise_target"] & (df["gt_match_status"] == "unique_match")

    df["sample_key"] = df.apply(
        lambda r: _sample_key(r.get("generated_question_id"), r.get("row_index")),
        axis=1,
    )
    lookup_audit_cols = [
        "generated_question_id",
        "row_index",
        "template_id",
        "scene_id",
        "ParentRuleID",
        "rule_figure_id",
        "gt_match_status",
        "gt_source_row_id",
        "gt_total_steps",
    ]
    keep_lookup_cols = [c for c in lookup_audit_cols if c in df.columns]
    if keep_lookup_cols:
        df[keep_lookup_cols].to_csv(GT_LOOKUP_AUDIT_CSV, index=False, encoding="utf-8-sig")

    return df, False


#### GT-Centric Auto Coverage Construction

Builds one row per GT step and attaches matched predicted step IDs/texts.
This GT-centric table is the primary review surface used for human overrides.


In [266]:
def _compute_auto_gt_coverage(df_cls: pd.DataFrame) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    rows = []
    pred_rule_map = {}
    sample_meta_rows = []

    for idx, row in df_cls.iterrows():
        generated_question_id = row.get("generated_question_id")
        row_index = row.get("row_index", idx)
        if _is_blank_value(row_index):
            row_index = idx
        sample_key = row.get("sample_key", _sample_key(generated_question_id, row_index))

        gt_text_full = row.get("ground_truth_answer", "")
        pred_text_full = row.get("model_answer", "")

        gt_steps_struct = _parse_reasoning_steps(gt_text_full)
        pred_steps_struct = _parse_reasoning_steps(pred_text_full)

        gt_steps = [_step_text_for_matching(st) for st in gt_steps_struct]
        pred_steps = [_step_text_for_matching(st) for st in pred_steps_struct]
        pred_labels = ["" if pd.isna(st.get("step_label", "")) else str(st.get("step_label", "")).strip() for st in pred_steps_struct]

        # Final-step guardrails: prevent final verdict from random matching with earlier reasoning steps.
        gt_final_idxs = _detect_final_step_indices(gt_steps_struct)
        pred_final_idxs = _detect_final_step_indices(pred_steps_struct)

        pred_rule_map[sample_key] = pred_steps

        sample_meta_rows.append(
            {
                "sample_key": sample_key,
                "generated_question_id": generated_question_id,
                "row_index": row_index,
                "template_id": row.get("template_id"),
                "scene_id": row.get("scene_id"),
                "parent_rule_id_norm": row.get("parent_rule_id_norm"),
                "figure_id_norm": row.get("figure_id_norm"),
                "gt_match_status": row.get("gt_match_status", ""),
                "question_text": row.get("question_text", ""),
                "full_ground_truth_answer": gt_text_full,
                "full_model_answer": pred_text_full,
                "model_name": row.get("model_name", MODEL_NAME),
            }
        )

        sim = _safe_sbert_pairwise_scores(pred_steps, gt_steps)
        for gt_i, gt_text in enumerate(gt_steps):
            col_scores_raw = sim[:, gt_i] if (sim.size > 0 and len(pred_steps) > 0) else np.array([], dtype=float)
            col_scores = _apply_final_step_incompatibility_guard(
                col_scores_raw,
                gt_i,
                gt_final_idxs,
                pred_final_idxs,
            )

            best_similarity = float(np.nanmax(col_scores)) if col_scores.size > 0 and not np.isnan(col_scores).all() else np.nan
            matched_pred_ids, coverage_structure, coverage_quality = _coverage_from_scores(
                col_scores,
                RULE_MATCH_THRESHOLD,
                RULE_STRONG_MATCH_THRESHOLD,
            )

            # Final-step fallback: if final GT step has no text-level match,
            # allow label-level final-step match and mark quality as wrong.
            if (gt_i in gt_final_idxs) and (len(matched_pred_ids) == 0):
                fallback_ids, fallback_label_sim = _match_final_step_by_label(
                    gt_steps_struct[gt_i],
                    pred_steps_struct,
                    pred_final_idxs,
                )
                if fallback_ids:
                    matched_pred_ids = fallback_ids
                    coverage_structure = "exact" if len(matched_pred_ids) == 1 else "split"
                    coverage_quality = "wrong"

            matched_pred_texts = [pred_steps[p] for p in matched_pred_ids]
            matched_pred_labels = [pred_labels[p] for p in matched_pred_ids]

            rows.append(
                {
                    "sample_key": sample_key,
                    "generated_question_id": generated_question_id,
                    "row_index": row_index,
                    "model_name": row.get("model_name", MODEL_NAME),
                    "question_text": row.get("question_text", ""),
                    "full_ground_truth_answer": gt_text_full,
                    "full_model_answer": pred_text_full,
                    "gt_step_id": int(gt_i),
                    "gt_step_position": int(gt_i + 1),
                    "gt_step_text": gt_text,
                    "gt_is_final_step": int(gt_i in gt_final_idxs),
                    "pred_final_step_ids": _serialize_ids(sorted(list(pred_final_idxs)), "P"),
                    "matched_pred_step_ids": _serialize_ids(matched_pred_ids, "P"),
                    "matched_pred_positions": _serialize_positions(matched_pred_ids),
                    "matched_pred_labels": _serialize_texts(matched_pred_labels),
                    "matched_pred_texts": _serialize_texts(matched_pred_texts),
                    "best_similarity": best_similarity,
                    "matched_pred_count": int(len(matched_pred_ids)),
                    "coverage_structure": coverage_structure,
                    "coverage_quality": coverage_quality,
                }
            )

    gt_df = pd.DataFrame(rows)
    if gt_df.empty:
        gt_df = pd.DataFrame(
            columns=[
                "sample_key",
                "generated_question_id",
                "row_index",
                "model_name",
                "question_text",
                "full_ground_truth_answer",
                "full_model_answer",
                "gt_step_id",
                "gt_step_position",
                "gt_step_text",
                "gt_is_final_step",
                "pred_final_step_ids",
                "matched_pred_step_ids",
                "matched_pred_positions",
                "matched_pred_labels",
                "matched_pred_texts",
                "best_similarity",
                "matched_pred_count",
                "coverage_structure",
                "coverage_quality",
            ]
        )

    sample_meta_df = pd.DataFrame(sample_meta_rows).drop_duplicates(subset=["sample_key"])
    return gt_df, pred_rule_map, sample_meta_df


#### Merge Human Review and Build Final GT Labels

Merges reviewer inputs (`human_coverage_*`, `human_best_pred_ids`, `human_notes`) into auto coverage rows.
Final labels (`final_coverage_*`, `final_best_pred_ids`) are computed with human values taking priority when provided.


In [267]:
def _merge_review_fields(auto_gt_df: pd.DataFrame, review_csv: str) -> pd.DataFrame:
    if auto_gt_df.empty:
        for c in ["human_coverage_structure", "human_coverage_quality", "human_best_pred_ids", "human_notes"]:
            if c not in auto_gt_df.columns:
                auto_gt_df[c] = ""
        return auto_gt_df

    out = auto_gt_df.copy()
    review_cols = ["human_coverage_structure", "human_coverage_quality", "human_best_pred_ids", "human_notes"]
    for c in review_cols:
        if c not in out.columns:
            out[c] = ""

    p = Path(review_csv)
    if not p.exists():
        return out

    prev = pd.read_csv(p)
    alias_map = {
        "human_coverage_structure": [
            "human_coverage_structure",
            "human_coverage_label",
            "Coverage Structure",
            "Human Coverage Label",
        ],
        "human_coverage_quality": [
            "human_coverage_quality",
            "Coverage Quality",
            "Human Coverage Quality",
        ],
        "human_best_pred_ids": [
            "human_best_pred_ids",
            "Human Best Pred IDs",
            "Human Best Read IDs",
            "Human Best Read Ids",
        ],
        "human_notes": ["human_notes", "Human Notes"],
    }
    for target, aliases in alias_map.items():
        if target not in prev.columns:
            prev[target] = ""
        if _is_blank_series(prev[target]).all():
            for alias in aliases:
                if alias in prev.columns:
                    prev[target] = prev[alias].fillna("")
                    if not _is_blank_series(prev[target]).all():
                        break

    # Normalize key-like numeric fields.
    for c in ["row_index", "gt_step_id", "gt_step_position"]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce").astype("Int64")
        if c in prev.columns:
            prev[c] = pd.to_numeric(prev[c], errors="coerce").astype("Int64")

    # Stage 1 (strict): model_name + generated_question_id + row_index + gt_step_id.
    strict_keys = [
        c for c in ["model_name", "generated_question_id", "row_index", "gt_step_id"]
        if c in out.columns and c in prev.columns
    ]
    if len(strict_keys) >= 2:
        prev_keep = prev[strict_keys + review_cols].drop_duplicates(subset=strict_keys)
        merged = out.merge(prev_keep, on=strict_keys, how="left", suffixes=("", "__prev"))
        for c in review_cols:
            cp = f"{c}__prev"
            if cp in merged.columns:
                mask = _is_blank_series(merged[c])
                merged.loc[mask, c] = merged.loc[mask, cp].fillna("")
                merged.drop(columns=[cp], inplace=True)
        out = merged

    # Stage 2 (fallback): model_name + generated_question_id + gt_step_position.
    # This keeps reviewed rows when gt_step_id shifts after parser tweaks.
    fallback_keys = [
        c for c in ["model_name", "generated_question_id", "gt_step_position"]
        if c in out.columns and c in prev.columns
    ]
    if len(fallback_keys) >= 2:
        prev_fb = prev[fallback_keys + review_cols].drop_duplicates(subset=fallback_keys)
        fb = out.merge(prev_fb, on=fallback_keys, how="left", suffixes=("", "__fb"))
        for c in review_cols:
            cfb = f"{c}__fb"
            if cfb in fb.columns:
                mask = _is_blank_series(fb[c])
                fb.loc[mask, c] = fb.loc[mask, cfb].fillna("")
                fb.drop(columns=[cfb], inplace=True)
        out = fb

    return out


def _finalize_gt_coverage(gt_df: pd.DataFrame) -> pd.DataFrame:
    """Finalize reviewed GT coverage labels without persisting redundant auto_* columns."""
    out = gt_df.copy()
    if out.empty:
        for c in [
            "final_coverage_structure",
            "final_coverage_quality",
            "final_best_pred_ids",
        ]:
            if c not in out.columns:
                out[c] = ""
        return out

    base_struct = out.get("coverage_structure", "").fillna("").astype(str).str.strip().str.lower()
    base_quality = out.get("coverage_quality", "").fillna("").astype(str).str.strip().str.lower()
    base_best = out.get("matched_pred_step_ids", "").fillna("").astype(str)

    human_struct = out.get("human_coverage_structure", "").fillna("").astype(str).str.strip().str.lower()
    human_quality = out.get("human_coverage_quality", "").fillna("").astype(str).str.strip().str.lower()
    human_best = out.get("human_best_pred_ids", "").fillna("").astype(str).str.strip()

    out["final_coverage_structure"] = np.where(human_struct != "", human_struct, base_struct)
    out["final_coverage_quality"] = np.where(human_quality != "", human_quality, base_quality)
    out["final_coverage_quality"] = np.where(out["final_coverage_structure"] == "missing", "", out["final_coverage_quality"])
    out["final_best_pred_ids"] = np.where(human_best != "", human_best, base_best)
    return out


#### Sample-Level Metrics and Aggregate Helpers

Computes per-sample counts/rates from final GT-centric labels and matched prediction IDs.
These outputs feed final comparison tables across models and reasoning steps.


In [268]:
def _order_preservation_from_sample(gt_sub: pd.DataFrame) -> tuple[float, int, float, int]:
    if gt_sub.empty:
        return np.nan, 0, np.nan, 0

    anchors = []
    for _, row in gt_sub.sort_values("gt_step_position").iterrows():
        pred_ids = _parse_pred_ids(row.get("final_best_pred_ids", ""))
        if not pred_ids:
            continue
        gt_pos = pd.to_numeric(row.get("gt_step_position", np.nan), errors="coerce")
        if pd.isna(gt_pos):
            continue
        anchor_pos = min(pred_ids) + 1  # 1-based position
        anchors.append((int(gt_pos), int(anchor_pos)))

    if len(anchors) < 2:
        strict_rate = float(np.mean([int(g == a) for g, a in anchors])) if anchors else np.nan
        return np.nan, 0, strict_rate, 0

    comparable_pairs = 0
    correctly_ordered_pairs = 0
    for i in range(len(anchors)):
        for j in range(i + 1, len(anchors)):
            gt_i, pred_i = anchors[i]
            gt_j, pred_j = anchors[j]
            if gt_i == gt_j:
                continue
            comparable_pairs += 1
            if (gt_i < gt_j and pred_i < pred_j) or (gt_i > gt_j and pred_i > pred_j):
                correctly_ordered_pairs += 1

    score = _safe_div(correctly_ordered_pairs, comparable_pairs)
    strict_match_rate = float(np.mean([int(g == a) for g, a in anchors])) if anchors else np.nan
    return score, int(max(0, comparable_pairs - correctly_ordered_pairs)), strict_match_rate, comparable_pairs


def _compute_sample_metrics_from_final(gt_final: pd.DataFrame, pred_rule_map: dict, sample_meta_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if sample_meta_df.empty:
        return pd.DataFrame(
            columns=[
                "sample_key",
                "generated_question_id",
                "row_index",
                "gt_total_steps",
                "pred_total_steps",
                "unique_matched_pred_count",
                "gt_covered_count",
                "gt_missing_count",
                "gt_split_count",
                "gt_weak_count",
                "pred_extra_count",
                "coverage_recall",
                "split_step_rate",
                "missing_step_rate",
                "weak_match_rate",
                "extra_step_rate",
                "order_preservation_score",
                "order_error_count",
                "strict_position_match_rate",
                "comparable_matched_pairs",
            ]
        )

    gt_grouped = gt_final.groupby("sample_key", dropna=False) if not gt_final.empty else {}
    for _, meta in sample_meta_df.iterrows():
        sample_key = meta["sample_key"]
        gt_sub = gt_grouped.get_group(sample_key).copy() if (not gt_final.empty and sample_key in gt_grouped.groups) else pd.DataFrame()

        gt_total_steps = int(len(gt_sub))
        if gt_total_steps > 0:
            gt_struct = gt_sub["final_coverage_structure"].astype(str).str.strip().str.lower()
            gt_quality = gt_sub["final_coverage_quality"].astype(str).str.strip().str.lower()
            gt_covered_count = int(gt_struct.isin(["exact", "split"]).sum())
            gt_missing_count = int((gt_struct == "missing").sum())
            gt_split_count = int((gt_struct == "split").sum())
            gt_weak_count = int(((gt_struct != "missing") & (gt_quality == "weak")).sum())
            # Extra-step counting is based on IDs used in final_best_pred_ids.
            # One predicted step may support multiple GT steps but counts once per sample.
            used_ids = {pid for v in gt_sub["final_best_pred_ids"].tolist() for pid in _parse_pred_ids(v)}
            order_score, order_error_count, strict_pos_rate, comparable_pairs = _order_preservation_from_sample(gt_sub)
        else:
            gt_covered_count = 0
            gt_missing_count = 0
            gt_split_count = 0
            gt_weak_count = 0
            used_ids = set()
            order_score, order_error_count, strict_pos_rate, comparable_pairs = np.nan, 0, np.nan, 0

        pred_total_steps = int(len(pred_rule_map.get(sample_key, [])))
        unique_matched_pred_count = int(len(used_ids))
        pred_extra_count = int(max(0, pred_total_steps - unique_matched_pred_count))

        rows.append(
            {
                "sample_key": sample_key,
                "generated_question_id": meta.get("generated_question_id"),
                "row_index": meta.get("row_index"),
                "gt_total_steps": gt_total_steps,
                "pred_total_steps": pred_total_steps,
                "unique_matched_pred_count": unique_matched_pred_count,
                "gt_covered_count": gt_covered_count,
                "gt_missing_count": gt_missing_count,
                "gt_split_count": gt_split_count,
                "gt_weak_count": gt_weak_count,
                "pred_extra_count": pred_extra_count,
                "coverage_recall": _safe_div(gt_covered_count, gt_total_steps),
                "split_step_rate": _safe_div(gt_split_count, gt_total_steps),
                "missing_step_rate": _safe_div(gt_missing_count, gt_total_steps),
                "weak_match_rate": _safe_div(gt_weak_count, gt_total_steps),
                "extra_step_rate": _safe_div(pred_extra_count, pred_total_steps),
                "order_preservation_score": order_score,
                "order_error_count": order_error_count,
                "strict_position_match_rate": strict_pos_rate,
                "comparable_matched_pairs": comparable_pairs,
            }
        )

    return pd.DataFrame(rows)

def _classification_rouge_per_rule(gt_final: pd.DataFrame, pred_rule_map: dict) -> float:
    if gt_final.empty:
        return np.nan

    scores = []
    for _, row in gt_final.iterrows():
        gt_text = row.get("gt_step_text", "")
        pred_ids = _parse_pred_ids(row.get("final_best_pred_ids", ""))
        pred_rules = pred_rule_map.get(row.get("sample_key"), [])
        selected = [pred_rules[i] for i in pred_ids if 0 <= i < len(pred_rules)]
        pred_text = " ".join(selected).strip()
        score = _rouge_l_f1(gt_text, pred_text)
        if not pd.isna(score):
            scores.append(float(score))
    return float(np.mean(scores)) if scores else np.nan


### Stepwise Reasoning Evaluation (Templates 41 and 42, Human-Review Flow)

#### Define Export Table Schemas

In [269]:
candidate_rows = []
pairwise_rows = []
gt_coverage_rows = []
sample_metrics_rows = []

candidate_columns = [
    "generated_question_id",
    "row_index",
    "model_name",
    "question_text",
    "pred_step_id",
    "pred_step_position",
    "pred_step_label",
    "pred_text",
    "matched_gt_id",
    "matched_gt_position",
    "matched_gt_text",
    "full_ground_truth_answer",
    "full_model_answer",
    "similarity_score",
    "triage_zone",
    "human_review_required",
    "candidate_flag",
    "auto_match",
]
pairwise_columns = [
    "generated_question_id",
    "row_index",
    "model_name",
    "question_text",
    "full_ground_truth_answer",
    "full_model_answer",
    "gt_step_id",
    "gt_step_position",
    "gt_is_final_step",
    "pred_final_step_ids",
    "gt_step_text",
    "pred_step_id",
    "pred_step_position",
    "pred_step_label",
    "pred_text",
    "similarity_score",
    "triage_zone",
    "human_review_required",
    "candidate_flag",
    "auto_match",
]
gt_coverage_columns = [
    "generated_question_id",
    "row_index",
    "model_name",
    "question_text",
    "full_ground_truth_answer",
    "full_model_answer",
    "gt_step_id",
    "gt_step_position",
    "gt_is_final_step",
    "pred_final_step_ids",
    "matched_pred_step_ids",
    "gt_step_text",
    "matched_pred_positions",
    "matched_pred_texts",
    "matched_pred_labels",
    "best_similarity",
    "triage_zone",
    "matched_pred_count",
    "coverage_structure",
    "coverage_quality",
    "human_coverage_structure",
    "human_coverage_quality",
    "human_best_pred_ids",
    "human_notes",
    "final_coverage_structure",
    "final_coverage_quality",
    "final_best_pred_ids",
]
sample_metrics_columns = [
    "generated_question_id",
    "row_index",
    "model_name",
    "question_text",
    "full_ground_truth_answer",
    "full_model_answer",
    "gt_total_steps",
    "pred_total_steps",
    "unique_matched_pred_count",
    "gt_covered_count",
    "gt_missing_count",
    "gt_split_count",
    "gt_weak_count",
    "pred_extra_count",
    "coverage_recall",
    "split_step_rate",
    "missing_step_rate",
    "weak_match_rate",
    "extra_step_rate",
    "order_preservation_score",
    "order_error_count",
    "strict_position_match_rate",
    "comparable_matched_pairs",
    "pair_auto_accept_count",
    "pair_uncertain_count",
    "pair_auto_reject_count",
]


#### Prepare Stepwise Review Tables (GT Lookup Applied)

In [270]:
# Main stepwise source of truth for this notebook run:
# load prepared rows with canonical GT reasoning attached via scene_id + parent_rule_id + figure_id.
df_eval, _missing_primary = _prepare_rule_eval_df(INPUT, model_name=MODEL_NAME)
if _missing_primary:
    raise FileNotFoundError(f"Primary input not found: {INPUT}")

rule_classification_mask = df_eval["_is_stepwise_target"]
rule_understanding_mask = ~rule_classification_mask

df_rule_classification = df_eval[rule_classification_mask].copy()
df_rule_understanding = df_eval[rule_understanding_mask].copy()

print("Prepared routing summary (after GT retrieval):")
print(f"- Stepwise subset (templates {sorted(STEPWISE_TEMPLATE_IDS)}): {len(df_rule_classification)} rows")
print(f"- Auto-eligible unique-match rows: {int(df_rule_classification['_is_auto_eligible'].sum())}")
print(f"- Manual-review queue rows: {int(len(df_rule_classification) - df_rule_classification['_is_auto_eligible'].sum())}")

for idx, row in df_rule_classification.iterrows():
    generated_question_id = row.get("generated_question_id")
    row_index_value = row.get("row_index", idx)
    if pd.isna(row_index_value):
        row_index_value = idx
    model_name_value = row.get(COL_MODEL_NAME, MODEL_NAME)
    question_text = "" if pd.isna(row.get("question_text", "")) else str(row.get("question_text", ""))
    full_ground_truth_answer = "" if pd.isna(row.get(COL_GT, "")) else str(row.get(COL_GT, ""))
    full_model_answer = "" if pd.isna(row.get(COL_PRED, "")) else str(row.get(COL_PRED, ""))

    gt_steps_struct = _parse_reasoning_steps(full_ground_truth_answer)
    pred_steps_struct = _parse_reasoning_steps(full_model_answer)

    gt_steps = [_step_text_for_matching(st) for st in gt_steps_struct]
    pred_steps = [_step_text_for_matching(st) for st in pred_steps_struct]
    pred_step_labels = ["" if pd.isna(st.get("step_label", "")) else str(st.get("step_label", "")).strip() for st in pred_steps_struct]

    sim = _sbert_pairwise_scores(pred_steps, gt_steps)

    # Raw audit matrix: pairwise similarity for each predicted/GT step pair.
    for pred_i, pred_text in enumerate(pred_steps):
        pred_label = pred_step_labels[pred_i] if pred_i < len(pred_step_labels) else ""
        for gt_i, gt_text in enumerate(gt_steps):
            score = float(sim[pred_i, gt_i]) if sim.size > 0 else np.nan
            pairwise_rows.append(
                {
                    "generated_question_id": generated_question_id,
                    "row_index": row_index_value,
                    "model_name": model_name_value,
                    "question_text": question_text,
                    "full_ground_truth_answer": full_ground_truth_answer,
                    "full_model_answer": full_model_answer,
                    "gt_step_id": gt_i,
                    "gt_step_position": gt_i + 1,
                    "gt_step_text": gt_text,
                    "pred_step_id": pred_i,
                    "pred_step_position": pred_i + 1,
                    "pred_step_label": pred_label,
                    "pred_text": pred_text,
                    "similarity_score": score,
                    "candidate_flag": int((not pd.isna(score)) and (score >= RULE_WEAK_MATCH_THRESHOLD)),
                    "auto_match": int((not pd.isna(score)) and (score >= RULE_MATCH_THRESHOLD)),
                }
            )

    # One-to-one assignment (pred -> gt) used only for candidate review table.
    pred_to_gt = _greedy_one_to_one_assignment(sim)
    if len(pred_steps) == 0:
        candidate_rows.append(
            {
                "generated_question_id": generated_question_id,
                "row_index": row_index_value,
                "model_name": model_name_value,
                "question_text": question_text,
                "pred_step_id": np.nan,
                "pred_step_position": np.nan,
                "pred_step_label": "",
                "pred_text": "",
                "matched_gt_id": np.nan,
                "matched_gt_position": np.nan,
                "matched_gt_text": "",
                "full_ground_truth_answer": full_ground_truth_answer,
                "full_model_answer": full_model_answer,
                "similarity_score": np.nan,
                "candidate_flag": 0,
                "auto_match": 0,
            }
        )
    else:
        for pred_i, pred_text in enumerate(pred_steps):
            pred_label = pred_step_labels[pred_i] if pred_i < len(pred_step_labels) else ""
            if pred_i in pred_to_gt:
                gt_i = pred_to_gt[pred_i]
                best_score = float(sim[pred_i, gt_i]) if sim.size > 0 else np.nan
                matched_gt_id = int(gt_i)
                matched_gt_position = int(gt_i + 1)
                matched_gt_text = gt_steps[gt_i]
            else:
                best_score = np.nan
                matched_gt_id = np.nan
                matched_gt_position = np.nan
                matched_gt_text = ""

            candidate_rows.append(
                {
                    "generated_question_id": generated_question_id,
                    "row_index": row_index_value,
                    "model_name": model_name_value,
                    "question_text": question_text,
                    "pred_step_id": pred_i,
                    "pred_step_position": pred_i + 1,
                    "pred_step_label": pred_label,
                    "pred_text": pred_text,
                    "matched_gt_id": matched_gt_id,
                    "matched_gt_position": matched_gt_position,
                    "matched_gt_text": matched_gt_text,
                    "full_ground_truth_answer": full_ground_truth_answer,
                    "full_model_answer": full_model_answer,
                    "similarity_score": best_score,
                    "candidate_flag": int((not pd.isna(best_score)) and (best_score >= RULE_WEAK_MATCH_THRESHOLD)),
                    "auto_match": int((not pd.isna(best_score)) and (best_score >= RULE_MATCH_THRESHOLD)),
                }
            )


Prepared routing summary (after GT retrieval):
- Stepwise subset (templates [41, 42]): 140 rows
- Auto-eligible unique-match rows: 4
- Manual-review queue rows: 136


#### Materialize Candidate / Pairwise / GT / Sample Tables

In [271]:
rule_classification_candidate_matches_df = pd.DataFrame(candidate_rows, columns=candidate_columns)
rule_classification_all_pairwise_df = pd.DataFrame(pairwise_rows, columns=pairwise_columns)
# GT-centric coverage/sample metrics are materialized in the adjudication step (single source of truth).
rule_classification_gt_coverage_summary_df = pd.DataFrame(gt_coverage_rows, columns=gt_coverage_columns)
rule_classification_sample_level_metrics_df = pd.DataFrame(sample_metrics_rows, columns=sample_metrics_columns)
rule_classification_pairwise_df = rule_classification_all_pairwise_df.copy()


#### Apply Triage and Human-Review Flags

In [272]:
# Three-zone triage for review artifacts.
if "similarity_score" in rule_classification_candidate_matches_df.columns:
    rule_classification_candidate_matches_df["triage_zone"] = rule_classification_candidate_matches_df["similarity_score"].apply(
        lambda x: _triage_zone(x, RULE_AUTO_ACCEPT_THRESHOLD, RULE_AUTO_REJECT_THRESHOLD)
    )
    rule_classification_candidate_matches_df["needs_human_review"] = (
        rule_classification_candidate_matches_df["triage_zone"].map(_needs_human_review)
    )

if "similarity_score" in rule_classification_all_pairwise_df.columns:
    rule_classification_all_pairwise_df["triage_zone"] = rule_classification_all_pairwise_df["similarity_score"].apply(
        lambda x: _triage_zone(x, RULE_AUTO_ACCEPT_THRESHOLD, RULE_AUTO_REJECT_THRESHOLD)
    )
    rule_classification_all_pairwise_df["needs_human_review"] = (
        rule_classification_all_pairwise_df["triage_zone"].map(_needs_human_review)
    )
    rule_classification_pairwise_df = rule_classification_all_pairwise_df.copy()

if "needs_human_review" in rule_classification_all_pairwise_df.columns:
    rule_classification_all_pairwise_df["human_review_required"] = (
        rule_classification_all_pairwise_df["needs_human_review"].fillna(0).astype(int)
    )
else:
    rule_classification_all_pairwise_df["human_review_required"] = 0

if "human_review_required" not in rule_classification_candidate_matches_df.columns and "needs_human_review" in rule_classification_candidate_matches_df.columns:
    rule_classification_candidate_matches_df["human_review_required"] = (
        rule_classification_candidate_matches_df["needs_human_review"].fillna(0).astype(int)
    )


#### Adjudication and Final Sample Metrics

In [273]:
# Adjudication for candidate table (review helper fields only).
if "needs_human_review" in rule_classification_candidate_matches_df.columns:
    rule_classification_candidate_matches_df["human_review_required"] = (
        rule_classification_candidate_matches_df["needs_human_review"].fillna(0).astype(int)
    )
else:
    rule_classification_candidate_matches_df["human_review_required"] = 0

rule_classification_candidate_matches_df["auto_label"] = rule_classification_candidate_matches_df.get("auto_match", 0)
for col in ["human_label", "human_gt_assignment", "human_pred_assignment"]:
    if col not in rule_classification_candidate_matches_df.columns:
        rule_classification_candidate_matches_df[col] = ""
    else:
        rule_classification_candidate_matches_df[col] = rule_classification_candidate_matches_df[col].fillna("")

_human_label_norm = rule_classification_candidate_matches_df["human_label"].astype(str).str.strip()
rule_classification_candidate_matches_df["final_label"] = np.where(
    _human_label_norm != "",
    _human_label_norm,
    rule_classification_candidate_matches_df["auto_label"].astype(str),
)

_human_gt_norm = rule_classification_candidate_matches_df["human_gt_assignment"].astype(str).str.strip()
rule_classification_candidate_matches_df["final_gt_assignment"] = np.where(
    _human_gt_norm != "",
    _human_gt_norm,
    rule_classification_candidate_matches_df.get("matched_gt_id"),
)

_human_pred_norm = rule_classification_candidate_matches_df["human_pred_assignment"].astype(str).str.strip()
rule_classification_candidate_matches_df["final_pred_assignment"] = np.where(
    _human_pred_norm != "",
    _human_pred_norm,
    rule_classification_candidate_matches_df.get("pred_step_id"),
)

# GT-centric path (single source of truth): auto coverage -> merge review -> final labels -> sample metrics.
df_rule_classification_eval = df_rule_classification[df_rule_classification["_is_auto_eligible"]].copy()
rule_classification_gt_auto_df, rule_classification_pred_rule_map, rule_classification_sample_meta_df = _compute_auto_gt_coverage(
    df_rule_classification_eval
)
rule_classification_gt_merged_df = _merge_review_fields(rule_classification_gt_auto_df, RULE_GT_COVERAGE_CSV)
rule_classification_gt_coverage_summary_df = _finalize_gt_coverage(rule_classification_gt_merged_df)

# Diagnostic: how many GT rows already contain any human review signal.
_review_cols = ["human_coverage_structure", "human_coverage_quality", "human_best_pred_ids", "human_notes"]
_review_cols = [x for x in _review_cols if x in rule_classification_gt_coverage_summary_df.columns]
if _review_cols:
    _review_mask = pd.Series([False] * len(rule_classification_gt_coverage_summary_df), index=rule_classification_gt_coverage_summary_df.index)
    for _c in _review_cols:
        _review_mask = _review_mask | (~_is_blank_series(rule_classification_gt_coverage_summary_df[_c]))
    print(f"Loaded human-reviewed GT steps carried forward: {int(_review_mask.sum())}")

if "best_similarity" in rule_classification_gt_coverage_summary_df.columns:
    rule_classification_gt_coverage_summary_df["triage_zone"] = rule_classification_gt_coverage_summary_df["best_similarity"].apply(
        lambda x: _triage_zone(x, RULE_AUTO_ACCEPT_THRESHOLD, RULE_AUTO_REJECT_THRESHOLD)
    )
    rule_classification_gt_coverage_summary_df["needs_human_review"] = (
        rule_classification_gt_coverage_summary_df["triage_zone"].map(_needs_human_review)
    )
else:
    rule_classification_gt_coverage_summary_df["triage_zone"] = ""
    rule_classification_gt_coverage_summary_df["needs_human_review"] = 0

# Keep automatic triage signal in full GT coverage export: 1 = review suggested, 0 = optional.
rule_classification_gt_coverage_summary_df["human_review_required"] = (
    rule_classification_gt_coverage_summary_df["needs_human_review"].fillna(0).astype(int)
)

# Keep only one review flag in working tables as well.
for _df_name in [
    "rule_classification_candidate_matches_df",
    "rule_classification_all_pairwise_df",
    "rule_classification_gt_coverage_summary_df",
]:
    _df = globals().get(_df_name)
    if isinstance(_df, pd.DataFrame) and "needs_human_review" in _df.columns:
        _df.drop(columns=["needs_human_review"], inplace=True)

rule_classification_sample_level_metrics_df = _compute_sample_metrics_from_final(
    rule_classification_gt_coverage_summary_df,
    rule_classification_pred_rule_map,
    rule_classification_sample_meta_df,
)

rule_classification_sample_level_metrics_df["model_name"] = MODEL_NAME
meta_cols = [
    "sample_key",
    "model_name",
    "template_id",
    "scene_id",
    "parent_rule_id_norm",
    "figure_id_norm",
    "gt_match_status",
    "question_text",
    "full_ground_truth_answer",
    "full_model_answer",
]
meta_cols = [c for c in meta_cols if c in rule_classification_sample_meta_df.columns]
rule_classification_sample_level_metrics_df = rule_classification_sample_level_metrics_df.merge(
    rule_classification_sample_meta_df[meta_cols],
    on="sample_key",
    how="left",
)
print(f"Auto-eligible rows (unique GT match, templates {sorted(STEPWISE_TEMPLATE_IDS)}): {len(df_rule_classification_eval)}")
print(f"Non-eligible rows for manual review: {len(df_rule_classification) - len(df_rule_classification_eval)}")


Loaded human-reviewed GT steps carried forward: 26
Auto-eligible rows (unique GT match, templates [41, 42]): 4
Non-eligible rows for manual review: 136


#### Sort, Export, and Reporting

In [274]:
for col in ["pair_auto_accept_count", "pair_uncertain_count", "pair_auto_reject_count"]:
    if col not in rule_classification_sample_level_metrics_df.columns:
        rule_classification_sample_level_metrics_df[col] = 0

if (
    not rule_classification_all_pairwise_df.empty
    and {"generated_question_id", "row_index", "triage_zone"}.issubset(rule_classification_all_pairwise_df.columns)
):
    zone_counts = (
        rule_classification_all_pairwise_df
        .groupby(["generated_question_id", "row_index", "triage_zone"], dropna=False)
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    zone_counts = zone_counts.rename(
        columns={
            "auto_accept": "pair_auto_accept_count",
            "uncertain": "pair_uncertain_count",
            "auto_reject": "pair_auto_reject_count",
        }
    )
    for col in ["pair_auto_accept_count", "pair_uncertain_count", "pair_auto_reject_count"]:
        if col not in zone_counts.columns:
            zone_counts[col] = 0
    keep_cols = [
        "generated_question_id",
        "row_index",
        "pair_auto_accept_count",
        "pair_uncertain_count",
        "pair_auto_reject_count",
    ]
    rule_classification_sample_level_metrics_df = (
        rule_classification_sample_level_metrics_df
        .drop(columns=["pair_auto_accept_count", "pair_uncertain_count", "pair_auto_reject_count"], errors="ignore")
        .merge(zone_counts[keep_cols], on=["generated_question_id", "row_index"], how="left")
    )

for col in ["pair_auto_accept_count", "pair_uncertain_count", "pair_auto_reject_count"]:
    rule_classification_sample_level_metrics_df[col] = (
        rule_classification_sample_level_metrics_df[col].fillna(0).astype(int)
    )

# Stable ordering for manual review.
sort_cols = [
    c
    for c in ["generated_question_id", "row_index", "pred_step_id"]
    if c in rule_classification_candidate_matches_df.columns
]
if sort_cols and not rule_classification_candidate_matches_df.empty:
    rule_classification_candidate_matches_df = (
        rule_classification_candidate_matches_df.sort_values(sort_cols).reset_index(drop=True)
    )

pair_sort_cols = [
    c
    for c in ["generated_question_id", "row_index", "pred_step_id", "gt_step_id"]
    if c in rule_classification_all_pairwise_df.columns
]
if pair_sort_cols and not rule_classification_all_pairwise_df.empty:
    rule_classification_all_pairwise_df = (
        rule_classification_all_pairwise_df.sort_values(pair_sort_cols).reset_index(drop=True)
    )
    rule_classification_pairwise_df = rule_classification_all_pairwise_df.copy()

gt_sort_cols = [c for c in ["generated_question_id", "row_index", "gt_step_id"] if c in rule_classification_gt_coverage_summary_df.columns]
if gt_sort_cols and not rule_classification_gt_coverage_summary_df.empty:
    rule_classification_gt_coverage_summary_df = (
        rule_classification_gt_coverage_summary_df.sort_values(gt_sort_cols).reset_index(drop=True)
    )

sample_sort_cols = [c for c in ["generated_question_id", "row_index"] if c in rule_classification_sample_level_metrics_df.columns]
if sample_sort_cols and not rule_classification_sample_level_metrics_df.empty:
    rule_classification_sample_level_metrics_df = (
        rule_classification_sample_level_metrics_df.sort_values(sample_sort_cols).reset_index(drop=True)
    )

# unique_matched_pred_count: unique predicted IDs from final_best_pred_ids across all GT rows in a sample.
# One predicted step may support multiple GT rows but is counted once as used for extra-step counting.



def _prepare_review_flag_export_df(df_in: pd.DataFrame) -> pd.DataFrame:
    out = df_in.copy()
    if "human_review_required" not in out.columns and "needs_human_review" in out.columns:
        out["human_review_required"] = out["needs_human_review"].fillna(0).astype(int)
    if "needs_human_review" in out.columns:
        out.drop(columns=["needs_human_review"], inplace=True)
    if "human_review_required" in out.columns:
        out = out.rename(columns={"human_review_required": "Human Review Required"})
    return out

# Candidate export: keep only one review-flag column.
rule_classification_candidate_export_df = _prepare_review_flag_export_df(rule_classification_candidate_matches_df)

rule_classification_candidate_export_df.to_csv(
    RULE_CLASSIFICATION_CANDIDATE_CSV,
    index=False,
    encoding="utf-8-sig",
)

# Pairwise export: keep only one review-flag column.
rule_classification_pairwise_export_df = _prepare_review_flag_export_df(rule_classification_all_pairwise_df)

rule_classification_pairwise_export_df.to_csv(
    RULE_ALL_PAIRWISE_CSV,
    index=False,
    encoding="utf-8-sig",
)

# Export full GT coverage table for human review across all rows.
rule_classification_gt_coverage_export_df = rule_classification_gt_coverage_summary_df.copy()
for _c in ["auto_coverage_structure", "auto_coverage_quality", "auto_best_pred_ids"]:
    if _c in rule_classification_gt_coverage_export_df.columns:
        rule_classification_gt_coverage_export_df.drop(columns=[_c], inplace=True)

# Keep only one review-flag column and put matched text close to GT step text.
rule_classification_gt_coverage_export_df = _prepare_review_flag_export_df(rule_classification_gt_coverage_export_df)

gt_preferred_order = [
    "generated_question_id",
    "row_index",
    "model_name",
    "question_text",
    "full_ground_truth_answer",
    "full_model_answer",
    "gt_step_id",
    "gt_step_position",
    "gt_is_final_step",
    "pred_final_step_ids",
    "matched_pred_step_ids",
    "gt_step_text",
    "matched_pred_positions",
    "matched_pred_texts",
    "matched_pred_labels",
    "best_similarity",
    "triage_zone",
    "Human Review Required",
    "matched_pred_count",
    "coverage_structure",
    "coverage_quality",
    "human_coverage_structure",
    "human_coverage_quality",
    "human_best_pred_ids",
    "human_notes",
    "final_coverage_structure",
    "final_coverage_quality",
    "final_best_pred_ids",
]
_gt_keep = [c for c in gt_preferred_order if c in rule_classification_gt_coverage_export_df.columns]
_gt_rest = [c for c in rule_classification_gt_coverage_export_df.columns if c not in _gt_keep]
rule_classification_gt_coverage_export_df = rule_classification_gt_coverage_export_df[_gt_keep + _gt_rest]

rule_classification_gt_coverage_export_df.to_csv(
    RULE_GT_COVERAGE_CSV,
    index=False,
    encoding="utf-8-sig",
)

rule_classification_sample_level_metrics_df.to_csv(
    RULE_SAMPLE_LEVEL_METRICS_CSV,
    index=False,
    encoding="utf-8-sig",
)

print(f"Stepwise reasoning rows (templates {sorted(STEPWISE_TEMPLATE_IDS)}): {len(df_rule_classification)}")
print(
    f"Thresholds: match={RULE_MATCH_THRESHOLD}, "
    f"auto_accept={RULE_AUTO_ACCEPT_THRESHOLD}, auto_reject={RULE_AUTO_REJECT_THRESHOLD}"
)
print(f"Saved: {RULE_CLASSIFICATION_CANDIDATE_CSV} ({len(rule_classification_candidate_matches_df)} rows)")
print(f"Saved: {RULE_ALL_PAIRWISE_CSV} ({len(rule_classification_all_pairwise_df)} rows)")
print(f"Saved: {RULE_GT_COVERAGE_CSV} ({len(rule_classification_gt_coverage_summary_df)} rows)")
print(f"Saved: {RULE_SAMPLE_LEVEL_METRICS_CSV} ({len(rule_classification_sample_level_metrics_df)} rows)")

if "triage_zone" in rule_classification_all_pairwise_df.columns and not rule_classification_all_pairwise_df.empty:
    print("Pairwise triage zones:")
    print(rule_classification_all_pairwise_df["triage_zone"].value_counts(dropna=False).to_string())

rule_classification_candidate_matches_df.head(20)


Stepwise reasoning rows (templates [41, 42]): 140
Thresholds: match=0.71, auto_accept=0.82, auto_reject=0.58
Saved: 3_2_to_3_5_step_candidate_matches.csv (815 rows)
Saved: 3_2_to_3_5_all_pairwise_similarity.csv (164 rows)
Saved: 3_2_to_3_5_gt_coverage_summary.csv (26 rows)
Saved: 3_2_to_3_5_sample_level_metrics.csv (4 rows)
Pairwise triage zones:
triage_zone
uncertain      78
auto_reject    76
auto_accept    10


,generated_question_id,row_index,model_name,question_text,pred_step_id,pred_step_position,pred_step_label,pred_text,matched_gt_id,matched_gt_position,...,human_review_required,candidate_flag,auto_match,auto_label,human_label,human_gt_assignment,human_pred_assignment,final_label,final_gt_assignment,final_pred_assignment
0,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,0.0,1.0,Identify the design in the first picture,Identify the design in the first picture Step ...,1.0,2.0,...,1,1,1,1,,,,1,1.0,0.0
1,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,1.0,2.0,Compare with Figure 88 (semi-ambulant WC refer...,Compare with Figure 88 (semi-ambulant WC refer...,0.0,1.0,...,0,1,1,1,,,,1,0.0,1.0
2,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,2.0,3.0,Check the Emerald (Toilet) Zone requirements,Check the Emerald (Toilet) Zone requirements S...,3.0,4.0,...,0,1,1,1,,,,1,3.0,2.0
3,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,3.0,4.0,Check the Purple (Hand-rinse Basin) Zone requi...,Check the Purple (Hand-rinse Basin) Zone requi...,4.0,5.0,...,0,1,1,1,,,,1,4.0,3.0
4,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,4.0,5.0,Assess overall room dimensions,Assess overall room dimensions Step 5: Assess ...,5.0,6.0,...,1,1,0,0,,,,0,5.0,4.0
5,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,5.0,6.0,Evaluate compliance,Evaluate compliance Step 6: Evaluate complianc...,6.0,7.0,...,1,1,0,0,,,,0,6.0,5.0
6,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,6.0,7.0,Final check,Final check Step 7: Final check.\nThe critical...,2.0,3.0,...,0,0,0,0,,,,0,2.0,6.0
7,17,1,Claude Opus 4.6,Is the design displayed in the first picture c...,0.0,1.0,Identify the basin in the plan,Identify the basin in the plan Step 1: Identif...,NaN,NaN,...,0,0,0,0,,,,0,NaN,0.0
8,17,1,Claude Opus 4.6,Is the design displayed in the first picture c...,1.0,2.0,Identify grabrails and fixtures,Identify grabrails and fixtures Step 2: Identi...,NaN,NaN,...,0,0,0,0,,,,0,NaN,1.0
9,17,1,Claude Opus 4.6,Is the design displayed in the first picture c...,2.0,3.0,Assess whether the basin has a height-adjustme...,Assess whether the basin has a height-adjustme...,NaN,NaN,...,0,0,0,0,,,,0,NaN,2.0


#### Non-Eligible Rows and GT-Retrieval Audit

In [275]:
# Non-stepwise routed subset is tracked here for audit/debug.

df_rule_understanding = df_eval[rule_understanding_mask].copy()

print(f"Non-stepwise routed rows: {len(df_rule_understanding)}")
if "gt_match_status" in df_eval.columns:
    print("GT retrieval statuses:")
    print(df_eval["gt_match_status"].value_counts(dropna=False).to_string())

Non-stepwise routed rows: 0
GT retrieval statuses:
gt_match_status
no_match        136
unique_match      4


In [276]:
print("Stepwise candidate table shape:", rule_classification_candidate_matches_df.shape)
display(rule_classification_candidate_matches_df.head(10))

print("Stepwise pairwise similarity table shape:", rule_classification_all_pairwise_df.shape)
display(rule_classification_all_pairwise_df.head(10))

print("Stepwise GT coverage summary table shape:", rule_classification_gt_coverage_summary_df.shape)
display(rule_classification_gt_coverage_summary_df.head(10))

print("Stepwise sample-level metrics table shape:", rule_classification_sample_level_metrics_df.shape)
display(rule_classification_sample_level_metrics_df.head(10))


Stepwise candidate table shape: (815, 25)


,generated_question_id,row_index,model_name,question_text,pred_step_id,pred_step_position,pred_step_label,pred_text,matched_gt_id,matched_gt_position,...,human_review_required,candidate_flag,auto_match,auto_label,human_label,human_gt_assignment,human_pred_assignment,final_label,final_gt_assignment,final_pred_assignment
0,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,0.0,1.0,Identify the design in the first picture,Identify the design in the first picture Step ...,1.0,2.0,...,1,1,1,1,,,,1,1.0,0.0
1,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,1.0,2.0,Compare with Figure 88 (semi-ambulant WC refer...,Compare with Figure 88 (semi-ambulant WC refer...,0.0,1.0,...,0,1,1,1,,,,1,0.0,1.0
2,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,2.0,3.0,Check the Emerald (Toilet) Zone requirements,Check the Emerald (Toilet) Zone requirements S...,3.0,4.0,...,0,1,1,1,,,,1,3.0,2.0
3,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,3.0,4.0,Check the Purple (Hand-rinse Basin) Zone requi...,Check the Purple (Hand-rinse Basin) Zone requi...,4.0,5.0,...,0,1,1,1,,,,1,4.0,3.0
4,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,4.0,5.0,Assess overall room dimensions,Assess overall room dimensions Step 5: Assess ...,5.0,6.0,...,1,1,0,0,,,,0,5.0,4.0
5,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,5.0,6.0,Evaluate compliance,Evaluate compliance Step 6: Evaluate complianc...,6.0,7.0,...,1,1,0,0,,,,0,6.0,5.0
6,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,6.0,7.0,Final check,Final check Step 7: Final check.\nThe critical...,2.0,3.0,...,0,0,0,0,,,,0,2.0,6.0
7,17,1,Claude Opus 4.6,Is the design displayed in the first picture c...,0.0,1.0,Identify the basin in the plan,Identify the basin in the plan Step 1: Identif...,NaN,NaN,...,0,0,0,0,,,,0,NaN,0.0
8,17,1,Claude Opus 4.6,Is the design displayed in the first picture c...,1.0,2.0,Identify grabrails and fixtures,Identify grabrails and fixtures Step 2: Identi...,NaN,NaN,...,0,0,0,0,,,,0,NaN,1.0
9,17,1,Claude Opus 4.6,Is the design displayed in the first picture c...,2.0,3.0,Assess whether the basin has a height-adjustme...,Assess whether the basin has a height-adjustme...,NaN,NaN,...,0,0,0,0,,,,0,NaN,2.0


Stepwise pairwise similarity table shape: (164, 20)


,generated_question_id,row_index,model_name,question_text,full_ground_truth_answer,full_model_answer,gt_step_id,gt_step_position,gt_is_final_step,pred_final_step_ids,gt_step_text,pred_step_id,pred_step_position,pred_step_label,pred_text,similarity_score,triage_zone,human_review_required,candidate_flag,auto_match
0,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,0,1,NaN,NaN,Identify the reference standard (Figure 88) Fi...,0,1,Identify the design in the first picture,Identify the design in the first picture Step ...,0.575488,auto_reject,0,0,0
1,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,1,2,NaN,NaN,Identify design elements and assess dimensiona...,0,1,Identify the design in the first picture,Identify the design in the first picture Step ...,0.810846,uncertain,1,1,1
2,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,2,3,NaN,NaN,Compare with Figure 88 (semi-ambulant WC refer...,0,1,Identify the design in the first picture,Identify the design in the first picture Step ...,0.556597,auto_reject,0,0,0
3,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,3,4,NaN,NaN,Assess the Emerald (Toilet) Activity Zone comp...,0,1,Identify the design in the first picture,Identify the design in the first picture Step ...,0.489108,auto_reject,0,0,0
4,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,4,5,NaN,NaN,Assess the Purple (Hand-rinse Basin) Activity ...,0,1,Identify the design in the first picture,Identify the design in the first picture Step ...,0.345987,auto_reject,0,0,0
5,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,5,6,NaN,NaN,Evaluate circulation space and potential zone ...,0,1,Identify the design in the first picture,Identify the design in the first picture Step ...,0.301253,auto_reject,0,0,0
6,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,6,7,NaN,NaN,Final assessment Step 7: Final assessment\n\nT...,0,1,Identify the design in the first picture,Identify the design in the first picture Step ...,0.526229,auto_reject,0,0,0
7,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,0,1,NaN,NaN,Identify the reference standard (Figure 88) Fi...,1,2,Compare with Figure 88 (semi-ambulant WC refer...,Compare with Figure 88 (semi-ambulant WC refer...,0.849072,auto_accept,0,1,1
8,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,1,2,NaN,NaN,Identify design elements and assess dimensiona...,1,2,Compare with Figure 88 (semi-ambulant WC refer...,Compare with Figure 88 (semi-ambulant WC refer...,0.573871,auto_reject,0,0,0
9,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,2,3,NaN,NaN,Compare with Figure 88 (semi-ambulant WC refer...,1,2,Compare with Figure 88 (semi-ambulant WC refer...,Compare with Figure 88 (semi-ambulant WC refer...,0.746805,uncertain,1,1,1


Stepwise GT coverage summary table shape: (26, 29)


,sample_key,generated_question_id,row_index,model_name,question_text,full_ground_truth_answer,full_model_answer,gt_step_id,gt_step_position,gt_step_text,...,coverage_quality,human_coverage_structure,human_coverage_quality,human_best_pred_ids,human_notes,final_coverage_structure,final_coverage_quality,final_best_pred_ids,triage_zone,human_review_required
0,16||0,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,0,1,Identify the reference standard (Figure 88) Fi...,...,weak,exact,weak,P_1,reviewed,exact,weak,P_1,auto_accept,0
1,16||0,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,1,2,Identify design elements and assess dimensiona...,...,weak,,,,reviewed,exact,weak,P_0,uncertain,1
2,16||0,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,2,3,Compare with Figure 88 (semi-ambulant WC refer...,...,weak,exact,weak,P_4,reviewed,exact,weak,P_4,uncertain,1
3,16||0,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,3,4,Assess the Emerald (Toilet) Activity Zone comp...,...,weak,exact,,P_2,reviewed,exact,weak,P_2,auto_accept,0
4,16||0,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,4,5,Assess the Purple (Hand-rinse Basin) Activity ...,...,weak,exact,,P_3,reviewed,exact,weak,P_3,auto_accept,0
5,16||0,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,5,6,Evaluate circulation space and potential zone ...,...,,exact,weak,P_4,reviewed,exact,weak,P_4,uncertain,1
6,16||0,16,0,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,6,7,Final assessment Step 7: Final assessment\n\nT...,...,wrong,exact,weak,P_6,reviewed,exact,weak,P_6,auto_reject,0
7,36||17,36,17,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the rule requirements and key...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,0,1,Identify the rule requirements and key complia...,...,weak,,,"P_2,P_4",reviewed,split,weak,"P_2,P_4",uncertain,1
8,36||17,36,17,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the rule requirements and key...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,1,2,Identify door types and locations in the asses...,...,weak,,,,reviewed,exact,weak,P_0,uncertain,1
9,36||17,36,17,Claude Opus 4.6,Is the design displayed in the first picture c...,Step 1: Identify the rule requirements and key...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,2,3,Use commonsense and domain knowledge to evalua...,...,,,,,reviewed,missing,,,auto_reject,0


Stepwise sample-level metrics table shape: (4, 33)


,sample_key,generated_question_id,row_index,gt_total_steps,pred_total_steps,unique_matched_pred_count,gt_covered_count,gt_missing_count,gt_split_count,gt_weak_count,...,scene_id,parent_rule_id_norm,figure_id_norm,gt_match_status,question_text,full_ground_truth_answer,full_model_answer,pair_auto_accept_count,pair_uncertain_count,pair_auto_reject_count
0,16||0,16,0,7,7,6,7,0,0,7,...,1,2.0,1.0,unique_match,Is the design displayed in the first picture c...,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,3,20,26
1,36||17,36,17,6,6,5,5,1,2,4,...,9,1.0,NaN,unique_match,Is the design displayed in the first picture c...,Step 1: Identify the rule requirements and key...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,3,21,12
2,50||25,50,25,6,5,5,5,1,0,4,...,11,1.0,NaN,unique_match,Is the design displayed in the first picture c...,Step 1: Identify the rule requirements and key...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,2,20,8
3,246||126,246,126,7,7,5,6,1,1,6,...,26,2.0,1.0,unique_match,Is the design displayed in the first picture c...,\nStep 1: Identify the reference standard (Fig...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...,2,17,30


In [277]:
print("Section sizes (stepwise routing):")
print(f"Stepwise subset (templates {sorted(STEPWISE_TEMPLATE_IDS)}):", len(df_rule_classification))
print("Non-stepwise subset:", len(df_rule_understanding))

Section sizes (stepwise routing):
Stepwise subset (templates [41, 42]): 140
Non-stepwise subset: 0


In [278]:
preview_cols = [
    "generated_question_id",
    "row_index",
    "template_id",
    "question_text",
    "scene_id",
    "ParentRuleID",
    "rule_figure_id",
    "gt_match_status",
    "ground_truth_answer",
    "model_answer",
]
show_cols = [c for c in preview_cols if c in df_eval.columns]

if show_cols:
    display(df_eval.loc[rule_classification_mask, show_cols].head(10))
else:
    print("No preview columns found in df_eval.")

,generated_question_id,row_index,template_id,question_text,scene_id,ParentRuleID,rule_figure_id,gt_match_status,ground_truth_answer,model_answer
0,16,0,41,Is the design displayed in the first picture c...,1,2.0,1.0,unique_match,Step 1: Identify the reference standard (Figur...,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...
1,17,1,41,Is the design displayed in the first picture c...,1,7.0,NaN,no_match,,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...
2,18,2,42,Is the design displayed in the first picture c...,1,NaN,1.0,no_match,,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...
3,19,3,42,Is the design displayed in the first picture c...,1,NaN,1.0,no_match,,Step-by-Step Reasoning:\r\n\r\nStep 1: Underst...
4,20,4,42,Is the design displayed in the first picture c...,1,NaN,1.0,no_match,,Step-by-Step Reasoning:\r\n\r\nStep 1: Underst...
5,21,5,42,Is the design displayed in the first picture c...,1,NaN,1.0,no_match,,Step-by-Step Reasoning:\r\n\r\nStep 1: Underst...
6,22,6,42,Is the design displayed in the first picture c...,1,NaN,1.0,no_match,,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...
7,23,7,42,Is the design displayed in the first picture c...,1,NaN,1.0,no_match,,Step-by-Step Reasoning:\r\n\r\nStep 1: Underst...
8,24,8,42,Is the design displayed in the first picture c...,1,NaN,1.0,no_match,,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...
9,25,9,42,Is the design displayed in the first picture c...,1,NaN,1.0,no_match,,Step-by-Step Reasoning:\r\n\r\nStep 1: Identif...


### Save Export Dictionary

In [279]:
def _empty_rule_understanding_section_metrics() -> dict:
    return {
        "overall Sentence-BERT similarity": np.nan,
        "Stepwise reasoning": {
            "ROUGE-L (avg per step)": np.nan,
            "Coverage Recall": np.nan,
            "Split Step Rate": np.nan,
            "Missing Step Rate": np.nan,
            "Weak Match Rate": np.nan,
            "Extra Step Rate": np.nan,
            "Order Preservation Score": np.nan,
            "Strict Position Match Rate": np.nan,
        },
        "dataset characteristics": {
            "stepwise_subset": {"n_questions": 0, "gt_steps": 0, "text": _dataset_text_classification(0, 0)},
            "eligible_unique_match": {"n_questions": 0},
            "manual_review_queue": {"n_questions": 0},
        },
    }


def _build_rule_understanding_section_metrics(
    df_stepwise: pd.DataFrame,
    df_eligible: pd.DataFrame,
    gt_final: pd.DataFrame,
    sample_metrics: pd.DataFrame,
    pred_rule_map: dict,
) -> dict:
    overall_sbert = _mean_sbert_similarity_local(df_stepwise)
    cls_rouge = _classification_rouge_per_rule(gt_final, pred_rule_map)

    coverage_recall = _aggregate_rate(sample_metrics, "gt_covered_count", "gt_total_steps")
    split_rate = _aggregate_rate(sample_metrics, "gt_split_count", "gt_total_steps")
    missing_rate = _aggregate_rate(sample_metrics, "gt_missing_count", "gt_total_steps")
    weak_rate = _aggregate_rate(sample_metrics, "gt_weak_count", "gt_total_steps")
    extra_rate = _aggregate_rate(sample_metrics, "pred_extra_count", "pred_total_steps")

    order_score = float(sample_metrics["order_preservation_score"].dropna().mean()) if (not sample_metrics.empty and "order_preservation_score" in sample_metrics.columns) else np.nan
    strict_pos = float(sample_metrics["strict_position_match_rate"].dropna().mean()) if (not sample_metrics.empty and "strict_position_match_rate" in sample_metrics.columns) else np.nan

    gt_steps_total = int(sample_metrics["gt_total_steps"].sum()) if (not sample_metrics.empty and "gt_total_steps" in sample_metrics.columns) else int(len(gt_final))
    ds_stepwise = {
        "n_questions": int(len(df_stepwise)),
        "gt_steps": gt_steps_total,
        "text": _dataset_text_classification(int(len(df_stepwise)), gt_steps_total),
    }
    ds_eligible = {"n_questions": int(len(df_eligible))}
    ds_manual = {"n_questions": int(max(0, len(df_stepwise) - len(df_eligible)))}

    return {
        "overall Sentence-BERT similarity": overall_sbert,
        "Stepwise reasoning": {
            "ROUGE-L (avg per step)": cls_rouge,
            "Coverage Recall": coverage_recall,
            "Split Step Rate": split_rate,
            "Missing Step Rate": missing_rate,
            "Weak Match Rate": weak_rate,
            "Extra Step Rate": extra_rate,
            "Order Preservation Score": order_score,
            "Strict Position Match Rate": strict_pos,
        },
        "dataset characteristics": {
            "stepwise_subset": ds_stepwise,
            "eligible_unique_match": ds_eligible,
            "manual_review_queue": ds_manual,
        },
    }


def _build_model_rule_understanding_bundle(model_name: str, input_path: str, use_review_csv: bool = False) -> dict:
    df_model, missing = _prepare_rule_eval_df(input_path, model_name=model_name)
    if missing:
        return {
            "model": model_name,
            "missing_input": True,
            "input_path": input_path,
            "section_metrics": _empty_rule_understanding_section_metrics(),
            "artifacts": {},
        }

    df_stepwise = df_model[df_model["_is_stepwise_target"]].copy()
    df_eligible = df_model[df_model["_is_auto_eligible"]].copy()

    gt_auto, pred_rule_map, sample_meta_df = _compute_auto_gt_coverage(df_eligible)
    if use_review_csv:
        gt_merged = _merge_review_fields(gt_auto, RULE_GT_COVERAGE_CSV)
    else:
        gt_merged = gt_auto.copy()
        for c in ["human_coverage_structure", "human_coverage_quality", "human_best_pred_ids", "human_notes"]:
            if c not in gt_merged.columns:
                gt_merged[c] = ""

    gt_final = _finalize_gt_coverage(gt_merged)
    sample_metrics = _compute_sample_metrics_from_final(gt_final, pred_rule_map, sample_meta_df)

    section_metrics = _build_rule_understanding_section_metrics(
        df_stepwise,
        df_eligible,
        gt_final,
        sample_metrics,
        pred_rule_map,
    )

    return {
        "model": model_name,
        "missing_input": False,
        "input_path": input_path,
        "section_metrics": section_metrics,
        "artifacts": {
            "df_stepwise": df_stepwise,
            "df_eligible": df_eligible,
            "gt_coverage_final": gt_final,
            "sample_level_metrics": sample_metrics,
        },
    }


# 3. Gemini 3 Pro Preview (Same Human-Eval Pipeline)

### Quick Routing Check by Template

In [280]:
RULE_UNDERSTANDING_MODEL_BUNDLES = {}
RULE_UNDERSTANDING_MODEL_METRICS = {}

if "MODEL_METRICS" not in globals() or not isinstance(MODEL_METRICS, dict):
    MODEL_METRICS = {}

for _model_name in RULE_MODEL_COLUMNS:
    _input_path = RULE_MODEL_INPUTS.get(_model_name, "")
    _use_review = _model_name == "Claude Opus 4.6"

    # Reuse already computed Claude reviewed artifacts to avoid recomputing the same path.
    _has_precomputed_claude = all(
        k in globals()
        for k in [
            "df_rule_classification",
            "rule_classification_gt_coverage_summary_df",
            "rule_classification_sample_level_metrics_df",
            "rule_classification_pred_rule_map",
        ]
    )
    if _model_name == "Claude Opus 4.6" and _has_precomputed_claude:
        _df_stepwise = globals()["df_rule_classification"].copy()
        _df_eligible = globals().get("df_rule_classification_eval", pd.DataFrame()).copy()
        if _df_eligible.empty and "_is_auto_eligible" in _df_stepwise.columns:
            _df_eligible = _df_stepwise[_df_stepwise["_is_auto_eligible"]].copy()
        _gt_final = globals()["rule_classification_gt_coverage_summary_df"].copy()
        _sample_metrics = globals()["rule_classification_sample_level_metrics_df"].copy()
        _pred_rule_map = globals()["rule_classification_pred_rule_map"]
        _bundle = {
            "model": _model_name,
            "missing_input": False,
            "input_path": _input_path,
            "section_metrics": _build_rule_understanding_section_metrics(
                _df_stepwise,
                _df_eligible,
                _gt_final,
                _sample_metrics,
                _pred_rule_map,
            ),
            "artifacts": {
                "df_stepwise": _df_stepwise,
                "df_eligible": _df_eligible,
                "gt_coverage_final": _gt_final,
                "sample_level_metrics": _sample_metrics,
            },
        }
    else:
        _bundle = _build_model_rule_understanding_bundle(
            _model_name,
            _input_path,
            use_review_csv=_use_review,
        )
    RULE_UNDERSTANDING_MODEL_BUNDLES[_model_name] = _bundle
    RULE_UNDERSTANDING_MODEL_METRICS[_model_name] = _bundle["section_metrics"]

    if _model_name not in MODEL_METRICS or not isinstance(MODEL_METRICS.get(_model_name), dict):
        MODEL_METRICS[_model_name] = {}
    MODEL_METRICS[_model_name][RULE_UNDERSTANDING_TITLE] = _bundle["section_metrics"]

CLAUDE_RULE_UNDERSTANDING_METRICS = RULE_UNDERSTANDING_MODEL_METRICS.get("Claude Opus 4.6")
GEMINI_RULE_UNDERSTANDING_METRICS = RULE_UNDERSTANDING_MODEL_METRICS.get("Gemini 3 Pro Preview")
GPT52_RULE_UNDERSTANDING_METRICS = RULE_UNDERSTANDING_MODEL_METRICS.get("GPT-5.2")

for _model_name in RULE_MODEL_COLUMNS:
    _b = RULE_UNDERSTANDING_MODEL_BUNDLES[_model_name]
    _ds = _b["section_metrics"]["dataset characteristics"]
    if _b.get("missing_input"):
        print(f"{_model_name}: missing {_b.get('input_path')}. Metrics left blank.")
    else:
        print(
            f"{_model_name}: "
            f"stepwise -> {_ds['stepwise_subset']['text']} | "
            f"eligible unique match -> {_ds['eligible_unique_match']['n_questions']} | "
            f"manual review queue -> {_ds['manual_review_queue']['n_questions']}"
        )

RULE_UNDERSTANDING_MODEL_METRICS


Claude Opus 4.6: stepwise -> Q = 140, GT steps = 26 | eligible unique match -> 4 | manual review queue -> 136
Gemini 3 Pro Preview: stepwise -> Q = 140, GT steps = 26 | eligible unique match -> 4 | manual review queue -> 136
GPT-5.2: stepwise -> Q = 140, GT steps = 26 | eligible unique match -> 4 | manual review queue -> 136


{'Claude Opus 4.6': {'overall Sentence-BERT similarity': 0.7840825468301773,
  'Stepwise reasoning': {'ROUGE-L (avg per step)': 0.2187341948989855,
   'Coverage Recall': 0.8846153846153846,
   'Split Step Rate': 0.11538461538461539,
   'Missing Step Rate': 0.11538461538461539,
   'Weak Match Rate': 0.8076923076923077,
   'Extra Step Rate': 0.16,
   'Order Preservation Score': 0.844047619047619,
   'Strict Position Match Rate': 0.1773809523809524},
  'dataset characteristics': {'stepwise_subset': {'n_questions': 140,
    'gt_steps': 26,
    'text': 'Q = 140, GT steps = 26'},
   'eligible_unique_match': {'n_questions': 4},
   'manual_review_queue': {'n_questions': 136}}},
 'Gemini 3 Pro Preview': {'overall Sentence-BERT similarity': 0.8474886864423752,
  'Stepwise reasoning': {'ROUGE-L (avg per step)': 0.12976688935372718,
   'Coverage Recall': 0.6153846153846154,
   'Split Step Rate': 0.34615384615384615,
   'Missing Step Rate': 0.38461538461538464,
   'Weak Match Rate': 0.5,
   'Extra 

## 9. Final Excel Summary Table

Builds a reviewer-facing summary across models and reasoning steps.
The summary combines automatic metrics and human-alignment diagnostics.


In [281]:
import pandas as pd
import numpy as np

FINAL_SUMMARY_COLUMNS = [
    "Reasoning step",
    "Primary automatic metrics",
    "Claude Opus 4.6",
    "Gemini 3 Pro Preview",
    "GPT-5.2",
    "Dataset characteristics",
    "Human alignment metrics",
]

STEP_ROW_SPECS = [
    (1, "Step 1. Task understanding / Rule parsing"),
    (2, "Step 2. Information grounding in design scene"),
    (3, "Step 3. Comparison of rule and scene"),
    (4, "Step 4. Logical reasoning"),
    (5, "Step 5. Conclusion"),
]


def _fmt(v, digits: int = 3) -> str:
    if v is None:
        return ""
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    try:
        return f"{float(v):.{digits}f}"
    except Exception:
        return str(v)


def _parse_id_set(value) -> set[int]:
    return set(_parse_pred_ids(value))


def _jaccard(a: set[int], b: set[int]) -> float:
    if not a and not b:
        return np.nan
    u = a | b
    if not u:
        return np.nan
    return float(len(a & b) / len(u))


def _reviewed_mask(df_gt: pd.DataFrame) -> pd.Series:
    if df_gt.empty:
        return pd.Series([], dtype=bool)
    human_cols = [c for c in [
        "human_coverage_structure",
        "human_coverage_quality",
        "human_best_pred_ids",
        "human_notes",
    ] if c in df_gt.columns]
    if not human_cols:
        return pd.Series([False] * len(df_gt), index=df_gt.index)
    mask = pd.Series([False] * len(df_gt), index=df_gt.index)
    for c in human_cols:
        mask = mask | (~_is_blank_series(df_gt[c]))
    return mask


def _alignment_summary(df_gt: pd.DataFrame) -> dict:
    if df_gt.empty:
        return {
            "matched_id_overlap": np.nan,
            "unchanged_percentage": np.nan,
            "n_reviewed_steps": 0,
            "n_reviewed_questions": 0,
        }

    reviewed = _reviewed_mask(df_gt)
    n_reviewed_steps = int(reviewed.sum())
    if n_reviewed_steps == 0:
        return {
            "matched_id_overlap": np.nan,
            "unchanged_percentage": np.nan,
            "n_reviewed_steps": 0,
            "n_reviewed_questions": 0,
        }

    sub = df_gt[reviewed].copy()

    unchanged_parts = []
    if {"coverage_structure", "final_coverage_structure"}.issubset(sub.columns):
        unchanged_parts.append((sub["coverage_structure"].astype(str).str.strip() == sub["final_coverage_structure"].astype(str).str.strip()).astype(float))
    if {"coverage_quality", "final_coverage_quality"}.issubset(sub.columns):
        unchanged_parts.append((sub["coverage_quality"].astype(str).str.strip() == sub["final_coverage_quality"].astype(str).str.strip()).astype(float))

    unchanged_pct = np.nan
    if unchanged_parts:
        all_unchanged = pd.concat(unchanged_parts, axis=1).mean(axis=1)
        unchanged_pct = float(all_unchanged.mean())

    overlaps = []
    if {"matched_pred_step_ids", "final_best_pred_ids"}.issubset(sub.columns):
        for _, r in sub.iterrows():
            a = _parse_id_set(r.get("matched_pred_step_ids", ""))
            b = _parse_id_set(r.get("final_best_pred_ids", ""))
            j = _jaccard(a, b)
            if not pd.isna(j):
                overlaps.append(float(j))
    overlap_mean = float(np.mean(overlaps)) if overlaps else np.nan

    n_reviewed_questions = int(sub["sample_key"].nunique()) if "sample_key" in sub.columns else 0

    return {
        "matched_id_overlap": overlap_mean,
        "unchanged_percentage": unchanged_pct,
        "n_reviewed_steps": n_reviewed_steps,
        "n_reviewed_questions": n_reviewed_questions,
    }


def _stage_metrics_from_gt(df_gt_stage: pd.DataFrame) -> dict:
    if df_gt_stage.empty:
        return {
            "sbert": np.nan,
            "rouge_l": np.nan,
            "coverage_recall": np.nan,
            "split_rate": np.nan,
            "weak_rate": np.nan,
            "n_steps": 0,
            "n_questions": 0,
        }

    sub = df_gt_stage.copy()
    n = len(sub)

    final_structure = sub.get("final_coverage_structure", pd.Series([""] * n)).astype(str).str.strip().str.lower()
    final_quality = sub.get("final_coverage_quality", pd.Series([""] * n)).astype(str).str.strip().str.lower()

    covered = final_structure.isin(["exact", "split"]).sum()
    split = (final_structure == "split").sum()
    weak = ((final_structure != "missing") & (final_quality == "weak")).sum()

    sbert = float(sub["best_similarity"].dropna().mean()) if "best_similarity" in sub.columns else np.nan

    rouge_scores = []
    if {"gt_step_text", "matched_pred_texts"}.issubset(sub.columns):
        for _, r in sub.iterrows():
            gt_text = "" if pd.isna(r.get("gt_step_text")) else str(r.get("gt_step_text"))
            pred_text = "" if pd.isna(r.get("matched_pred_texts")) else str(r.get("matched_pred_texts"))
            if gt_text.strip() and pred_text.strip():
                rouge_scores.append(_rouge_l_f1(gt_text, pred_text))
    rouge_l = float(np.nanmean(rouge_scores)) if rouge_scores else np.nan

    n_questions = int(sub["sample_key"].nunique()) if "sample_key" in sub.columns else 0

    return {
        "sbert": sbert,
        "rouge_l": rouge_l,
        "coverage_recall": _safe_div(covered, n),
        "split_rate": _safe_div(split, n),
        "weak_rate": _safe_div(weak, n),
        "n_steps": int(n),
        "n_questions": n_questions,
    }


def _render_stage_model_value(stage_metrics: dict, overall_metrics: dict, step_idx: int) -> str:
    # Step 5 is conclusion-focused: keep final answer accuracy unavailable in this notebook context.
    if step_idx == 5:
        return "Accuracy=n/a | SBERT=" + _fmt(stage_metrics.get("sbert")) + " | ROUGE-L=" + _fmt(stage_metrics.get("rouge_l"))

    return (
        "SBERT=" + _fmt(stage_metrics.get("sbert"))
        + " | ROUGE-L=" + _fmt(stage_metrics.get("rouge_l"))
        + " | Coverage Recall=" + _fmt(stage_metrics.get("coverage_recall"))
        + " | Split Rate=" + _fmt(stage_metrics.get("split_rate"))
        + " | Weak Match Rate=" + _fmt(stage_metrics.get("weak_rate"))
    )


def _render_overall_model_value(section_metrics: dict) -> str:
    stepwise = section_metrics.get("Stepwise reasoning", {}) if isinstance(section_metrics, dict) else {}
    return (
        "SBERT=" + _fmt(section_metrics.get("overall Sentence-BERT similarity"))
        + " | ROUGE-L=" + _fmt(stepwise.get("ROUGE-L (avg per step)"))
        + " | Coverage Recall=" + _fmt(stepwise.get("Coverage Recall"))
        + " | Split Rate=" + _fmt(stepwise.get("Split Step Rate"))
        + " | Missing Step Rate=" + _fmt(stepwise.get("Missing Step Rate"))
        + " | Extra Step Rate=" + _fmt(stepwise.get("Extra Step Rate"))
        + " | Weak Match Rate=" + _fmt(stepwise.get("Weak Match Rate"))
        + " | Order Preservation=" + _fmt(stepwise.get("Order Preservation Score"))
    )


def _primary_metrics_text(step_idx: int) -> str:
    if step_idx == 2:
        return "SBERT Similarity; ROUGE-L; METEOR (optional); Coverage Recall; Split Rate; Extra Step Rate; Weak Match Rate; order_preservation_score"
    if step_idx == 4:
        return "SBERT Similarity; METEOR (optional); Coverage Recall; Split Rate; Extra Step Rate; Weak Match Rate; order_preservation_score"
    if step_idx == 5:
        return "Accuracy (primary); optional conclusion-level text metrics"
    return "SBERT Similarity; ROUGE-L; Coverage Recall; Split Rate; Extra Step Rate; Weak Match Rate; order_preservation_score"


def _first_non_blank(values: list[str]) -> str:
    for v in values:
        if isinstance(v, str) and v.strip():
            return v
    return ""


### 9.1 Human Alignment Metrics

Human alignment is computed only on reviewed GT rows.

Definitions:
- `matched_id_overlap`: Jaccard overlap between auto matched IDs (`matched_pred_step_ids`) and final reviewed IDs (`final_best_pred_ids`), averaged over reviewed rows.
- `unchanged_percentage`: share of reviewed rows where auto labels remained unchanged after adjudication (`coverage_structure` and `coverage_quality`).
- `n_reviewed_steps`: number of GT step rows with any human-review signal.
- `n_questions`: number of unique questions represented by reviewed rows.


In [282]:
# Human alignment metrics from reviewed GT rows.
# A row is considered reviewed if any human override column is filled,
# including human_notes (marker-only review without override is allowed).

HUMAN_ALIGNMENT_BY_MODEL = {}
_human_alignment_rows = []

for _model_name in RULE_MODEL_COLUMNS:
    _bundle = RULE_UNDERSTANDING_MODEL_BUNDLES.get(_model_name, {})
    _art = _bundle.get("artifacts", {}) if isinstance(_bundle, dict) else {}
    _gt_final = _art.get("gt_coverage_final", pd.DataFrame())

    _steps = {}
    for _step_idx, _ in STEP_ROW_SPECS:
        _sub = (
            _gt_final[_gt_final["gt_step_position"] == _step_idx].copy()
            if (isinstance(_gt_final, pd.DataFrame) and (not _gt_final.empty) and "gt_step_position" in _gt_final.columns)
            else pd.DataFrame()
        )
        _steps[_step_idx] = _alignment_summary(_sub)

    _overall = _alignment_summary(_gt_final if isinstance(_gt_final, pd.DataFrame) else pd.DataFrame())
    HUMAN_ALIGNMENT_BY_MODEL[_model_name] = {"steps": _steps, "overall": _overall}

    for _step_idx, _step_label in STEP_ROW_SPECS:
        _a = _steps.get(_step_idx, {})
        _human_alignment_rows.append(
            {
                "model": _model_name,
                "scope": f"step_{_step_idx}",
                "label": _step_label,
                "matched_id_overlap": _a.get("matched_id_overlap"),
                "unchanged_percentage": _a.get("unchanged_percentage"),
                "n_reviewed_steps": int(_a.get("n_reviewed_steps", 0) or 0),
                "n_questions": int(_a.get("n_reviewed_questions", 0) or 0),
            }
        )

    _human_alignment_rows.append(
        {
            "model": _model_name,
            "scope": "overall",
            "label": "Overall rate (per question)",
            "matched_id_overlap": _overall.get("matched_id_overlap"),
            "unchanged_percentage": _overall.get("unchanged_percentage"),
            "n_reviewed_steps": int(_overall.get("n_reviewed_steps", 0) or 0),
            "n_questions": int(_overall.get("n_reviewed_questions", 0) or 0),
        }
    )

human_alignment_metrics_df = pd.DataFrame(_human_alignment_rows)
display(human_alignment_metrics_df)


,model,scope,label,matched_id_overlap,unchanged_percentage,n_reviewed_steps,n_questions
0,Claude Opus 4.6,step_1,Step 1. Task understanding / Rule parsing,0.625000,0.625000,4,4
1,Claude Opus 4.6,step_2,Step 2. Information grounding in design scene,1.000000,1.000000,4,4
2,Claude Opus 4.6,step_3,Step 3. Comparison of rule and scene,0.500000,1.000000,4,4
3,Claude Opus 4.6,step_4,Step 4. Logical reasoning,0.666667,0.625000,4,4
4,Claude Opus 4.6,step_5,Step 5. Conclusion,0.583333,0.625000,4,4
5,Claude Opus 4.6,overall,Overall rate (per question),0.717391,0.769231,26,4
6,Gemini 3 Pro Preview,step_1,Step 1. Task understanding / Rule parsing,NaN,NaN,0,0
7,Gemini 3 Pro Preview,step_2,Step 2. Information grounding in design scene,NaN,NaN,0,0
8,Gemini 3 Pro Preview,step_3,Step 3. Comparison of rule and scene,NaN,NaN,0,0
9,Gemini 3 Pro Preview,step_4,Step 4. Logical reasoning,NaN,NaN,0,0


### 9.2 Build Final Summary Rows

Creates one row per reasoning step plus one overall row.
Each model column is rendered from step metrics and human-alignment summaries.


In [283]:
# Build per-model stage payloads once (reused for table and dictionary export).
model_stage_payload = {}
for model_name in RULE_MODEL_COLUMNS:
    bundle = RULE_UNDERSTANDING_MODEL_BUNDLES.get(model_name, {})
    sec = RULE_UNDERSTANDING_MODEL_METRICS.get(model_name, {}) if isinstance(RULE_UNDERSTANDING_MODEL_METRICS, dict) else {}
    artifacts = bundle.get("artifacts", {}) if isinstance(bundle, dict) else {}
    gt_final = artifacts.get("gt_coverage_final", pd.DataFrame())

    stage_stats = {}
    stage_alignment = {}
    _pre = HUMAN_ALIGNMENT_BY_MODEL.get(model_name, {}) if isinstance(globals().get("HUMAN_ALIGNMENT_BY_MODEL", {}), dict) else {}
    _pre_steps = _pre.get("steps", {}) if isinstance(_pre, dict) else {}
    for step_idx, _ in STEP_ROW_SPECS:
        sub = gt_final[gt_final["gt_step_position"] == step_idx].copy() if (isinstance(gt_final, pd.DataFrame) and (not gt_final.empty) and "gt_step_position" in gt_final.columns) else pd.DataFrame()
        stage_stats[step_idx] = _stage_metrics_from_gt(sub)
        stage_alignment[step_idx] = _pre_steps.get(step_idx, _alignment_summary(sub))

    overall_alignment = _pre.get("overall", _alignment_summary(gt_final if isinstance(gt_final, pd.DataFrame) else pd.DataFrame()))

    model_stage_payload[model_name] = {
        "section_metrics": sec,
        "stage_stats": stage_stats,
        "stage_alignment": stage_alignment,
        "overall_alignment": overall_alignment,
    }


final_summary_rows = []
for step_idx, step_label in STEP_ROW_SPECS:
    row = {
        "Reasoning step": step_label,
        "Primary automatic metrics": _primary_metrics_text(step_idx),
    }

    ds_values = []
    align_values = []
    for model_name in RULE_MODEL_COLUMNS:
        payload = model_stage_payload.get(model_name, {})
        stage_metrics = payload.get("stage_stats", {}).get(step_idx, {})
        section_metrics = payload.get("section_metrics", {})
        row[model_name] = _render_stage_model_value(stage_metrics, section_metrics, step_idx)

        n_steps = stage_metrics.get("n_steps", 0)
        n_q = stage_metrics.get("n_questions", 0)
        ds_values.append(f"n_steps={n_steps}; n_questions={n_q}")

        a = payload.get("stage_alignment", {}).get(step_idx, {})
        align_values.append(
            "matched_id_overlap=" + _fmt(a.get("matched_id_overlap"))
            + "; unchanged_percentage=" + _fmt(a.get("unchanged_percentage"))
            + "; n_reviewed_steps=" + str(int(a.get("n_reviewed_steps", 0) or 0))
        )

    # Dataset and alignment columns are shown once per row (Claude-first fallback).
    row["Dataset characteristics"] = _first_non_blank(ds_values)
    row["Human alignment metrics"] = _first_non_blank(align_values)
    final_summary_rows.append(row)


overall_row = {
    "Reasoning step": "Overall rate (per question)",
    "Primary automatic metrics": "SBERT Similarity; ROUGE-L; Coverage Recall; Split Rate; Missing Step Rate; Extra Step Rate; Weak Match Rate; order_preservation_score",
}

overall_ds_values = []
overall_align_values = []
for model_name in RULE_MODEL_COLUMNS:
    sec = RULE_UNDERSTANDING_MODEL_METRICS.get(model_name, {}) if isinstance(RULE_UNDERSTANDING_MODEL_METRICS, dict) else {}
    overall_row[model_name] = _render_overall_model_value(sec)

    ds = sec.get("dataset characteristics", {}) if isinstance(sec, dict) else {}
    stepwise_ds = ds.get("stepwise_subset", {}) if isinstance(ds, dict) else {}
    overall_ds_values.append(
        f"n_steps={int(stepwise_ds.get('gt_steps', 0) or 0)}; n_questions={int(stepwise_ds.get('n_questions', 0) or 0)}"
    )

    a = model_stage_payload.get(model_name, {}).get("overall_alignment", {})
    overall_align_values.append(
        "matched_id_overlap=" + _fmt(a.get("matched_id_overlap"))
        + "; unchanged_percentage=" + _fmt(a.get("unchanged_percentage"))
        + "; n_questions=" + str(int(a.get("n_reviewed_questions", 0) or 0))
    )

overall_row["Dataset characteristics"] = _first_non_blank(overall_ds_values)
overall_row["Human alignment metrics"] = _first_non_blank(overall_align_values)
final_summary_rows.append(overall_row)


rule_understanding_summary_df = pd.DataFrame(final_summary_rows, columns=FINAL_SUMMARY_COLUMNS)
rule_understanding_summary_df.to_csv(RULE_UNDERSTANDING_SUMMARY_CSV, index=False, encoding="utf-8-sig")
print("Saved:", RULE_UNDERSTANDING_SUMMARY_CSV)
print("Title:", RULE_UNDERSTANDING_TITLE)

STEPWISE_EXPORT_DICTIONARY = {
    "title": RULE_UNDERSTANDING_TITLE,
    "sheet_name": "Final Excel Summary",
    "model_columns": RULE_MODEL_COLUMNS,
    "summary_columns": FINAL_SUMMARY_COLUMNS,
    "step_row_specs": [{"step_index": i, "label": lbl} for i, lbl in STEP_ROW_SPECS],
    "model_metrics": RULE_UNDERSTANDING_MODEL_METRICS,
    "model_stage_payload": model_stage_payload,
    "human_alignment_metrics": HUMAN_ALIGNMENT_BY_MODEL if isinstance(globals().get("HUMAN_ALIGNMENT_BY_MODEL", {}), dict) else {},
    "human_alignment_metrics_table": human_alignment_metrics_df.to_dict(orient="records") if "human_alignment_metrics_df" in globals() else [],
    "summary_rows": final_summary_rows,
    "summary_table": rule_understanding_summary_df.to_dict(orient="records"),
    "artifact_paths": {
        "pairwise_csv": RULE_ALL_PAIRWISE_CSV,
        "gt_coverage_csv": RULE_GT_COVERAGE_CSV,
        "sample_metrics_csv": RULE_SAMPLE_LEVEL_METRICS_CSV,
        "lookup_audit_csv": GT_LOOKUP_AUDIT_CSV,
        "summary_csv": RULE_UNDERSTANDING_SUMMARY_CSV,
        "summary_xlsx": RULE_UNDERSTANDING_SUMMARY_XLSX,
    },
}

rule_understanding_summary_df


Saved: 3_2_to_3_5_stepwise_summary_table.csv
Title: Stepwise Reasoning Evaluation (3_2 to 3_5)


,Reasoning step,Primary automatic metrics,Claude Opus 4.6,Gemini 3 Pro Preview,GPT-5.2,Dataset characteristics,Human alignment metrics
0,Step 1. Task understanding / Rule parsing,SBERT Similarity; ROUGE-L; Coverage Recall; Sp...,SBERT=0.834 | ROUGE-L=0.263 | Coverage Recall=...,SBERT=0.823 | ROUGE-L=0.224 | Coverage Recall=...,SBERT=0.838 | ROUGE-L=0.294 | Coverage Recall=...,n_steps=4; n_questions=4,matched_id_overlap=0.625; unchanged_percentage...
1,Step 2. Information grounding in design scene,SBERT Similarity; ROUGE-L; METEOR (optional); ...,SBERT=0.789 | ROUGE-L=0.236 | Coverage Recall=...,SBERT=0.705 | ROUGE-L=0.177 | Coverage Recall=...,SBERT=0.644 | ROUGE-L=0.132 | Coverage Recall=...,n_steps=4; n_questions=4,matched_id_overlap=1.000; unchanged_percentage...
2,Step 3. Comparison of rule and scene,SBERT Similarity; ROUGE-L; Coverage Recall; Sp...,SBERT=0.596 | ROUGE-L=0.243 | Coverage Recall=...,SBERT=0.567 | ROUGE-L=0.198 | Coverage Recall=...,SBERT=0.475 | ROUGE-L= | Coverage Recall=0.000...,n_steps=4; n_questions=4,matched_id_overlap=0.500; unchanged_percentage...
3,Step 4. Logical reasoning,SBERT Similarity; METEOR (optional); Coverage ...,SBERT=0.837 | ROUGE-L=0.237 | Coverage Recall=...,SBERT=0.805 | ROUGE-L=0.204 | Coverage Recall=...,SBERT=0.777 | ROUGE-L=0.221 | Coverage Recall=...,n_steps=4; n_questions=4,matched_id_overlap=0.667; unchanged_percentage...
4,Step 5. Conclusion,Accuracy (primary); optional conclusion-level ...,Accuracy=n/a | SBERT=0.835 | ROUGE-L=0.198,Accuracy=n/a | SBERT=0.746 | ROUGE-L=0.146,Accuracy=n/a | SBERT=0.654 | ROUGE-L=0.150,n_steps=4; n_questions=4,matched_id_overlap=0.583; unchanged_percentage...
5,Overall rate (per question),SBERT Similarity; ROUGE-L; Coverage Recall; Sp...,SBERT=0.784 | ROUGE-L=0.219 | Coverage Recall=...,SBERT=0.847 | ROUGE-L=0.130 | Coverage Recall=...,SBERT=0.854 | ROUGE-L=0.122 | Coverage Recall=...,n_steps=26; n_questions=140,matched_id_overlap=0.717; unchanged_percentage...


### 9.3 Export Final Excel Summary

Exports the final summary table to `.xlsx` for reviewer and reporting workflows.


In [284]:
import pandas as pd
from pathlib import Path

xlsx_path = Path(RULE_UNDERSTANDING_SUMMARY_XLSX)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    rule_understanding_summary_df.to_excel(
        writer,
        sheet_name="Final Excel Summary",
        index=False,
    )

print(f"Saved: {xlsx_path}")
display(rule_understanding_summary_df)


Saved: 3_2_to_3_5_stepwise_summary_table.xlsx


,Reasoning step,Primary automatic metrics,Claude Opus 4.6,Gemini 3 Pro Preview,GPT-5.2,Dataset characteristics,Human alignment metrics
0,Step 1. Task understanding / Rule parsing,SBERT Similarity; ROUGE-L; Coverage Recall; Sp...,SBERT=0.834 | ROUGE-L=0.263 | Coverage Recall=...,SBERT=0.823 | ROUGE-L=0.224 | Coverage Recall=...,SBERT=0.838 | ROUGE-L=0.294 | Coverage Recall=...,n_steps=4; n_questions=4,matched_id_overlap=0.625; unchanged_percentage...
1,Step 2. Information grounding in design scene,SBERT Similarity; ROUGE-L; METEOR (optional); ...,SBERT=0.789 | ROUGE-L=0.236 | Coverage Recall=...,SBERT=0.705 | ROUGE-L=0.177 | Coverage Recall=...,SBERT=0.644 | ROUGE-L=0.132 | Coverage Recall=...,n_steps=4; n_questions=4,matched_id_overlap=1.000; unchanged_percentage...
2,Step 3. Comparison of rule and scene,SBERT Similarity; ROUGE-L; Coverage Recall; Sp...,SBERT=0.596 | ROUGE-L=0.243 | Coverage Recall=...,SBERT=0.567 | ROUGE-L=0.198 | Coverage Recall=...,SBERT=0.475 | ROUGE-L= | Coverage Recall=0.000...,n_steps=4; n_questions=4,matched_id_overlap=0.500; unchanged_percentage...
3,Step 4. Logical reasoning,SBERT Similarity; METEOR (optional); Coverage ...,SBERT=0.837 | ROUGE-L=0.237 | Coverage Recall=...,SBERT=0.805 | ROUGE-L=0.204 | Coverage Recall=...,SBERT=0.777 | ROUGE-L=0.221 | Coverage Recall=...,n_steps=4; n_questions=4,matched_id_overlap=0.667; unchanged_percentage...
4,Step 5. Conclusion,Accuracy (primary); optional conclusion-level ...,Accuracy=n/a | SBERT=0.835 | ROUGE-L=0.198,Accuracy=n/a | SBERT=0.746 | ROUGE-L=0.146,Accuracy=n/a | SBERT=0.654 | ROUGE-L=0.150,n_steps=4; n_questions=4,matched_id_overlap=0.583; unchanged_percentage...
5,Overall rate (per question),SBERT Similarity; ROUGE-L; Coverage Recall; Sp...,SBERT=0.784 | ROUGE-L=0.219 | Coverage Recall=...,SBERT=0.847 | ROUGE-L=0.130 | Coverage Recall=...,SBERT=0.854 | ROUGE-L=0.122 | Coverage Recall=...,n_steps=26; n_questions=140,matched_id_overlap=0.717; unchanged_percentage...


### 9.4 Export Dictionary Payload

Stores structured metadata about generated artifacts, summary columns, and per-model payloads.
Useful for reproducibility and downstream integration checks.


In [285]:
# Final Excel summary payload for downstream Excel mapping.
STEPWISE_EXPORT_DICTIONARY


{'title': 'Stepwise Reasoning Evaluation (3_2 to 3_5)',
 'sheet_name': 'Final Excel Summary',
 'model_columns': ['Claude Opus 4.6', 'Gemini 3 Pro Preview', 'GPT-5.2'],
 'summary_columns': ['Reasoning step',
  'Primary automatic metrics',
  'Claude Opus 4.6',
  'Gemini 3 Pro Preview',
  'GPT-5.2',
  'Dataset characteristics',
  'Human alignment metrics'],
 'step_row_specs': [{'step_index': 1,
   'label': 'Step 1. Task understanding / Rule parsing'},
  {'step_index': 2, 'label': 'Step 2. Information grounding in design scene'},
  {'step_index': 3, 'label': 'Step 3. Comparison of rule and scene'},
  {'step_index': 4, 'label': 'Step 4. Logical reasoning'},
  {'step_index': 5, 'label': 'Step 5. Conclusion'}],
 'model_metrics': {'Claude Opus 4.6': {'overall Sentence-BERT similarity': 0.7840825468301773,
   'Stepwise reasoning': {'ROUGE-L (avg per step)': 0.2187341948989855,
    'Coverage Recall': 0.8846153846153846,
    'Split Step Rate': 0.11538461538461539,
    'Missing Step Rate': 0.11538

In [286]:
# Quick check of exported final summary artifacts.
print("CSV:", RULE_UNDERSTANDING_SUMMARY_CSV)
print("XLSX:", RULE_UNDERSTANDING_SUMMARY_XLSX)


CSV: 3_2_to_3_5_stepwise_summary_table.csv
XLSX: 3_2_to_3_5_stepwise_summary_table.xlsx
